# Paper 3 XAI — v4 3×10 Multi-Cell Notebook

Notebook ini menjalankan eksperimen XAI Paper 3 secara bertahap di JupyterLab. Struktur eksperimen tetap satu evidence source, tetapi dieksekusi melalui beberapa stage dengan checkpoint/resume agar aman untuk eksperimen berat.

**Mode awal:** `RUN_MODE = "probe"` untuk 1 dataset × 1 seed × 1 fold. Setelah probe lolos, ubah menjadi `RUN_MODE = "final"`, restart kernel, lalu jalankan kembali dari atas.

**Stage:** setup → preflight/materialize → training → baseline → evaluation → export → audit.

Catatan: KernelSHAP dievaluasi pada seluruh sampel terpilih di setiap prosedur, bukan dummy subset. Sampel real-anomaly/counterfactual dipilih maksimum `cf_per_subtype` detected anomalies per subtype per fold.


In [ ]:
# =============================================================================
# v4-3x10 — MULTI-CELL JUPYTERLAB NOTEBOOK, ISOLATED OUTPUT
#   Stage 1 TRAIN     : SNN-AE, 3 seeds x 10 folds x 2 datasets  (= 60 models)
#   Stage 2 BASELINE  : XAI baselines from validation-normal only
#   Stage 3 EVALUATE  : final evidence run (faithfulness, counterfactual,
#                       SARED, suppression, stability, sensitivity,
#                       frozen-decoder intervention)
#
# Stitched verbatim from:
#   1-full-sdda-v2.py            (training + baseline; folds extended 0..9)
#   4-paper3-faithfulness-v2.py  (v3 evaluation; SHAP already full eval)
#
# SHAP OPTION 1 (full eval): KernelSHAP is evaluated on the full sample set
#   (faith_n samples in faithfulness, cf_per_subtype detected anomalies per subtype
#   for real-anomaly/counterfactual, and stability_n samples in stability). No dummy subset.
#
# SUPERVISION FIXES carried over + added:
#   [5] 33-feature schema audit + per-dataset feature hash
#   [6] SARED per-feature thresholds (te, td computed per column)
#   [7] local_explanations / per_attack_topk / feature_behavior for ALL methods
#   [8] counterfactual curves persisted (counterfactual_curves.npz)
#   [9] all-pairs Wilcoxon + Holm over 11 metrics  <-- EXPANDED in this file
#       (file 4 only covered log_audc + restricted_k)
#
# Determinism env replicated from file 4. Restart kernel when changing RUN_MODE.
# Resume is existence-based (train/baseline) + per-job done.json (eval).
# =============================================================================
import os, sys

if "tensorflow" in sys.modules:
    raise RuntimeError("Restart kernel, then run this cell once.")
os.environ.update(
    {
        "TF_CPP_MIN_LOG_LEVEL": "2",
        "TF_ENABLE_ONEDNN_OPTS": "0",
        "TF_DETERMINISTIC_OPS": "1",
        "TF_CUDNN_DETERMINISTIC": "1",
        "CUDA_VISIBLE_DEVICES": "-1",
        "TF_NUM_INTRAOP_THREADS": "4",
        "TF_NUM_INTEROP_THREADS": "4",
        "OMP_NUM_THREADS": "4",
        "PYTHONHASHSEED": "42",
    }
)

import gc, json, time, hashlib, platform, warnings
from pathlib import Path
from datetime import datetime
from itertools import combinations

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import joblib
import tensorflow as tf
import shap, sklearn, scipy
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, Dense, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler, QuantileTransformer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)
from sklearn.neighbors import NearestNeighbors
from scipy.stats import rankdata, spearmanr, wilcoxon

warnings.filterwarnings("ignore")
tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(4)
tf.keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()

# =============================================================================
# CONFIG
# =============================================================================
# --- Choose run size. Use "probe" first, then "final" after timing is acceptable. ---
# probe  : 1 dataset x 1 seed x 1 fold, written to paper3-faithfulness-v4-3x10-probe
# final  : 2 datasets x 3 seeds x 10 folds, written to paper3-faithfulness-v4-3x10
RUN_MODE = "final"  # change to "final" for final evidence run

CFG = {
    "protocol": "v4-3x10",
    "output": "paper3-faithfulness-v4-3x10",  # isolated; models + eval live here
    "v83": "0-snn-ae-unified-v8-3-fair-benchmark",  # splits + tuned hparams
    "datasets": ["edge_iiotset", "ciciot2023"],
    "seeds": [0, 1, 2],
    "folds": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],  # 3x10, matches Paper 1 CV
    # ---- data / materialize (from file 1) ----
    "data_path": {
        "edge_iiotset": "data/dataset-from-pcap/edge-iiotset_bidir.parquet",
        "ciciot2023": "data/dataset-from-pcap/ciciot2023_bidir.parquet",
    },
    "label_column": {"edge_iiotset": "label", "ciciot2023": "label"},
    "normal_class": {"edge_iiotset": "Normal", "ciciot2023": "Benign"},
    "drop_features": {
        "edge_iiotset": ["label", "label_type", "flow_id", "packet_index", "frame.time_epoch"],
        "ciciot2023": ["label", "label_type", "flow_id", "packet_index", "frame.time_epoch"],
    },
    "scaler_type": {"edge_iiotset": "quantile", "ciciot2023": "minmax"},
    "cap_per_subtype": 50000,
    "dim": 33,
    "seed": 42,
    "pb": 4096,
    # ---- training (from file 1) ----
    "val_ratio": 0.20,
    "operating_percentile": 99,
    "epochs": 50,
    "batch_size": 512,
    "patience": 8,
    "grad_clip": 1.0,
    "latent_dim": 32,
    # ---- faithfulness / eval (from file 4) ----
    "faith_n": 100,
    "m_features": 3,
    "magnitudes": [2.0, 4.0, 6.0],
    "shap_bg": 50,
    "shap_nsamples": 200,
    "runtime_n": 20,
    "runtime_repeats": 3,
    "cf_per_subtype": 10,
    "reference_pool": 2000,
    "sared_k": 2.0,
    "sared_pct": 75,
    "alpha": 0.5,
    "supp_cal_n": 2000,
    "supp_per_subtype": 2000,
    "feature_behavior_normal_n": 500,
    "feature_behavior_topk": 3,
    "stability_n": 20,
    "stability_eps": [0.01, 0.05, 0.1],
    "stability_repeats": 3,
    "shap_self_repeats": 5,
    "sared_k_grid": [1.5, 2.0, 2.5],
    "sared_pct_grid": [70, 75, 80],
    "alpha_grid": [0.25, 0.5, 1.0],
    "run_frozen_decoder_intervention": True,
    "decoder_widths": [32, 64],
    "decoder_epochs": 30,
    "decoder_patience": 5,
    "decoder_batch": 512,
    "decoder_learning_rate": 0.001,
    "decoder_train_n": 100000,
    "bootstrap": 10000,
}

if RUN_MODE not in {"probe", "final"}:
    raise ValueError("RUN_MODE must be 'probe' or 'final'")
if RUN_MODE == "probe":
    CFG["output"] = "paper3-faithfulness-v4-3x10-probe"
    CFG["datasets"] = ["edge_iiotset"]
    CFG["seeds"] = [0]
    CFG["folds"] = [0]
    print(">>> PROBE MODE: 1 dataset x 1 seed x 1 fold (schema + timing probe)")
else:
    CFG["output"] = "paper3-faithfulness-v4-3x10"
    print(">>> FINAL MODE: 2 datasets x 3 seeds x 10 folds")
CFG["run_mode"] = RUN_MODE

OUT = Path(CFG["output"])
SRC = OUT  # models trained into OUT; evaluator reads from OUT
V83 = Path(CFG["v83"])
RH = hashlib.sha256(json.dumps(CFG, sort_keys=True).encode()).hexdigest()[:16]
DIRS = {
    k: OUT / k
    for k in ["data", "models", "baselines", "jobs", "raw", "tables", "metadata", "figure_sources"]
}
for p in [OUT, *DIRS.values()]:
    p.mkdir(parents=True, exist_ok=True)
EPS = 1e-12


def now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def savejson(p, x):
    p = Path(p)
    q = p.with_suffix(p.suffix + ".tmp")
    with open(q, "w", encoding="utf-8") as f:
        json.dump(x, f, indent=2, allow_nan=False)
    os.replace(q, p)


def load_json(path, default=None):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {} if default is None else default


def stable_hash_array(arr):
    h = hashlib.sha256()
    h.update(np.ascontiguousarray(arr).view(np.uint8))
    return h.hexdigest()


mp = OUT / "metadata/manifest.json"
if mp.exists() and load_json(mp).get("run_hash") != RH:
    raise RuntimeError(
        "Output directory holds another configuration; change CFG['output'] "
        "or delete the directory. Probe and final mode use separate output folders."
    )
savejson(
    mp,
    {
        "run_hash": RH,
        "config": CFG,
        "created_at": now(),
        "versions": {
            "python": platform.python_version(),
            "tensorflow": tf.__version__,
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "sklearn": sklearn.__version__,
            "scipy": scipy.__version__,
            "shap": shap.__version__,
        },
        "devices": [str(x) for x in tf.config.list_physical_devices()],
    },
)



## Cell 2 — SpikingLayer, model builder, preflight

In [ ]:
# =============================================================================
# SPIKING LAYER (identical to v8.3 / file 1 / file 4 for .keras compatibility)
# =============================================================================
@tf.keras.utils.register_keras_serializable()
class SpikingLayer(Layer):
    def __init__(
        self,
        units,
        threshold=0.5,
        decay=0.7,
        surrogate_scale=10.0,
        time_steps=3,
        return_sequences=False,
        **kw,
    ):
        super().__init__(**kw)
        self.units = int(units)
        self.threshold = float(threshold)
        self.decay = float(decay)
        self.surrogate_scale = float(surrogate_scale)
        self.time_steps = int(time_steps)
        self.return_sequences = bool(return_sequences)

    def build(self, input_shape):
        self.input_dim = int(input_shape[-1])
        self.w = self.add_weight(
            shape=(self.input_dim, self.units),
            initializer=tf.keras.initializers.HeNormal(),
            trainable=True,
            name="weights",
        )
        self.b = self.add_weight(
            shape=(self.units,), initializer="zeros", trainable=True, name="bias"
        )
        super().build(input_shape)

    @tf.custom_gradient
    def spike_function(self, membrane, threshold):
        spikes = tf.cast(tf.greater(membrane, threshold), tf.float32)

        def grad(dy):
            diff = membrane - threshold
            sg = self.surrogate_scale / tf.square(
                self.surrogate_scale * tf.abs(diff) + 1.0
            )
            return dy * sg, None

        return spikes, grad

    def call(self, inputs, training=None):
        bs = tf.shape(inputs)[0]
        membrane = tf.zeros((bs, self.units), dtype=inputs.dtype)
        spikes = []
        is_seq = len(inputs.shape) == 3
        for t in range(self.time_steps):
            cur_in = inputs[:, t, :] if is_seq else inputs
            cur = tf.matmul(cur_in, self.w) + self.b
            cur = cur / tf.sqrt(tf.cast(tf.shape(cur_in)[-1], tf.float32))
            membrane = self.decay * membrane + cur
            spk = self.spike_function(membrane, self.threshold)
            membrane = membrane - spk * self.threshold
            spikes.append(spk)
        allspk = tf.stack(spikes, axis=1)
        return allspk if self.return_sequences else tf.reduce_mean(allspk, axis=1)

    def get_config(self):
        c = super().get_config()
        c.update(
            {
                "units": self.units,
                "threshold": self.threshold,
                "decay": self.decay,
                "surrogate_scale": self.surrogate_scale,
                "time_steps": self.time_steps,
                "return_sequences": self.return_sequences,
            }
        )
        return c


CUSTOM = {"SpikingLayer": SpikingLayer, "Custom>SpikingLayer": SpikingLayer}
METHODS = ["SDDA_z", "SDDA_KL", "SDDA_RW", "TSA_adapted", "RawError", "KernelSHAP", "Random"]


def make_scaler(ds):
    if CFG["scaler_type"][ds] == "quantile":
        return QuantileTransformer(
            n_quantiles=1000,
            output_distribution="normal",
            random_state=CFG["seed"],
            subsample=200000,
        )
    return MinMaxScaler()


def build_snn_ae(input_dim, cfg, name="SNN_AE"):
    L = int(cfg["latent_dim"])
    inp = Input(shape=(input_dim,), name="input")
    x = BatchNormalization(name="batch_norm")(inp)
    x = SpikingLayer(
        L,
        threshold=cfg["threshold"],
        decay=cfg["decay"],
        surrogate_scale=cfg["surrogate_scale"],
        time_steps=cfg["time_steps"],
        return_sequences=True,
        name="encoder",
    )(x)
    x = SpikingLayer(
        L,
        threshold=cfg["threshold"],
        decay=cfg["decay"],
        surrogate_scale=cfg["surrogate_scale"],
        time_steps=cfg["time_steps"],
        return_sequences=False,
        name="decoder",
    )(x)
    out = Dense(input_dim, activation="linear", name="output")(x)
    return Model(inp, out, name=name)


# =============================================================================
# PREFLIGHT + tuned hparams
# =============================================================================
def preflight():
    missing = []
    for ds in CFG["datasets"]:
        if not os.path.exists(CFG["data_path"][ds]):
            missing.append(CFG["data_path"][ds])
        if not (V83 / "splits" / f"{ds}_holdout_eval_split.npz").exists():
            missing.append(str(V83 / "splits" / f"{ds}_holdout_eval_split.npz"))
        for si in CFG["seeds"]:
            p = V83 / "splits" / f"{ds}_cv_seed{si}_fold_indices.npz"
            if not p.exists():
                missing.append(str(p))
    hp = V83 / "tuning" / "best_hparams.json"
    if not hp.exists():
        missing.append(str(hp))
    if missing:
        raise RuntimeError("PREFLIGHT FAILED, missing assets:\n" + "\n".join(missing))
    best = load_json(hp)
    # verify 10-fold keys exist in v8.3 CV splits (fold3..9 required for 3x10)
    for ds in CFG["datasets"]:
        if f"{ds}|SNN_AE" not in best:
            raise RuntimeError(f"Tuned SNN_AE hparams missing for {ds}")
        for si in CFG["seeds"]:
            zz = np.load(V83 / "splits" / f"{ds}_cv_seed{si}_fold_indices.npz")
            for f in CFG["folds"]:
                if f"fold{f}_trainval" not in zz or f"fold{f}_test" not in zz:
                    raise RuntimeError(
                        f"{ds} seed{si}: fold{f} missing in v8.3 CV split "
                        f"(3x10 needs folds {CFG['folds']})"
                    )
    print("[P0] Preflight OK")
    return best


def snn_cfg(ds, best):
    c = dict(best[f"{ds}|SNN_AE"]["best_params"])
    c["latent_dim"] = CFG["latent_dim"]
    c["time_steps"] = 3
    return c




## Cell 3 — Materialize dataset dan split loader

In [ ]:
# =============================================================================
# MATERIALIZE (memory-safe; identical selection to v8.3; reuse if present)
# =============================================================================
def materialize(ds):
    npz_path = DIRS["data"] / f"{ds}_balanced.npz"
    if npz_path.exists():
        z = np.load(npz_path, allow_pickle=True)
        return z["X"], z["y"], z["label_type"], [str(x) for x in z["feature_cols"]]
    path = CFG["data_path"][ds]
    lab, ncls, cap, rs = (
        CFG["label_column"][ds],
        CFG["normal_class"][ds],
        CFG["cap_per_subtype"],
        CFG["seed"],
    )
    print(f"[PM][{ds}] streaming labels from parquet")
    dfl = pd.read_parquet(
        path,
        columns=(
            [lab, "label_type"]
            if "label_type" in pq.ParquetFile(path).schema_arrow.names
            else [lab]
        ),
    )
    if "label_type" not in dfl.columns:
        dfl["label_type"] = dfl[lab].astype(str)
    df_n = dfl[dfl[lab] == ncls]
    df_a = dfl[dfl[lab] != ncls]
    assert len(df_n) > 0 and len(df_a) > 0, f"{ds}: empty class after filter"
    rng = np.random.RandomState(rs)
    parts = []
    for subtype in sorted(df_a["label_type"].astype(str).unique()):
        sub = df_a[df_a["label_type"].astype(str) == subtype]
        take = min(cap, len(sub))
        parts.append(sub.loc[rng.choice(sub.index, size=take, replace=False)])
    df_a_bal = pd.concat(parts, axis=0)
    n_normal = min(len(df_a_bal), len(df_n))
    df_n_bal = df_n.loc[rng.choice(df_n.index, size=n_normal, replace=False)]
    combined = pd.concat([df_n_bal, df_a_bal], axis=0).sample(frac=1.0, random_state=rs)
    y = (combined[lab] != ncls).astype(int).values
    label_type = combined["label_type"].astype(str).values
    pos_map = pd.Series(np.arange(len(dfl)), index=dfl.index)
    positions = pos_map.loc[combined.index].values.astype(np.int64)
    del dfl, df_n, df_a, df_a_bal, df_n_bal, parts
    gc.collect()
    pf = pq.ParquetFile(path)
    schema = pf.schema_arrow
    drop = set(CFG["drop_features"][ds])
    feat_cols = [
        f.name
        for f in schema
        if f.name not in drop
        and (pa.types.is_integer(f.type) or pa.types.is_floating(f.type))
    ]
    assert (
        len(feat_cols) == CFG["dim"]
    ), f"{ds}: {len(feat_cols)} numeric features, expected {CFG['dim']}: {feat_cols}"
    order = np.argsort(positions)
    sorted_pos = positions[order]
    X_sorted = np.empty((len(positions), len(feat_cols)), dtype=np.float32)
    off = cursor = 0
    for rg in range(pf.num_row_groups):
        n_rg = pf.metadata.row_group(rg).num_rows
        hi = cursor
        while hi < len(sorted_pos) and sorted_pos[hi] < off + n_rg:
            hi += 1
        if hi > cursor:
            tbl = pf.read_row_group(rg, columns=feat_cols)
            local = (sorted_pos[cursor:hi] - off).astype(np.int64)
            arr = tbl.take(pa.array(local)).to_pandas().to_numpy(dtype=np.float64)
            X_sorted[cursor:hi] = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0).astype(
                np.float32
            )
            del tbl, arr
        cursor = hi
        off += n_rg
        if rg % 20 == 0:
            gc.collect()
    assert cursor == len(sorted_pos), f"{ds}: row gather incomplete"
    X = np.empty_like(X_sorted)
    X[order] = X_sorted
    del X_sorted
    gc.collect()
    z = np.load(V83 / "splits" / f"{ds}_holdout_eval_split.npz", allow_pickle=True)
    if "y_hash" in z:
        assert str(z["y_hash"]) == stable_hash_array(y.astype(np.int64)) or str(
            z["y_hash"]
        ) == stable_hash_array(y), f"{ds}: y_hash mismatch vs v8.3"
        print(f"[PM][{ds}] y_hash verified against v8.3")
    np.savez_compressed(
        npz_path, X=X, y=y, label_type=label_type, feature_cols=np.array(feat_cols)
    )
    print(f"[PM][{ds}] balanced subset saved: {len(y):,} rows")
    return X, y, label_type, feat_cols


def load_splits(ds):
    z = np.load(V83 / "splits" / f"{ds}_holdout_eval_split.npz", allow_pickle=True)
    tune_idx, eval_idx = z["tune_idx"], z["eval_idx"]
    assert len(np.intersect1d(tune_idx, eval_idx)) == 0, "tune/eval overlap"
    folds = {}
    for si in CFG["seeds"]:
        zz = np.load(V83 / "splits" / f"{ds}_cv_seed{si}_fold_indices.npz")
        folds[si] = {f: (zz[f"fold{f}_trainval"], zz[f"fold{f}_test"]) for f in CFG["folds"]}
    return tune_idx, eval_idx, folds


def trainval_normal_split(y_tv, si, f):
    normal_local = np.where(y_tv == 0)[0]
    rng = np.random.RandomState(CFG["seed"] + 1000 * si + f)
    rng.shuffle(normal_local)
    split = int((1.0 - CFG["val_ratio"]) * len(normal_local))
    tr, vl = normal_local[:split], normal_local[split:]
    assert len(tr) > 0 and len(vl) > 0
    return tr, vl




## Cell 4 — Stage 1 function: train one fold

In [ ]:
# =============================================================================
# STAGE 1 — TRAIN (existence-based resume)
# =============================================================================
def model_dir(ds, si, f):
    p = DIRS["models"] / ds / f"seed{si}_fold{f}"
    p.mkdir(parents=True, exist_ok=True)
    return p


def train_one(ds, si, f, X_eval, y_eval, folds, cfg):
    jd = model_dir(ds, si, f)
    if (jd / "model.keras").exists() and (jd / "detection_sanity.json").exists():
        return
    tv, te = folds[si][f]
    X_tv, y_tv = X_eval[tv], y_eval[tv]
    X_te, y_te = X_eval[te], y_eval[te]
    tr, vl = trainval_normal_split(y_tv, si, f)
    assert np.all(y_tv[tr] == 0) and np.all(y_tv[vl] == 0), "LEAK: attack in train/val"
    scaler = make_scaler(ds)
    Xtr = scaler.fit_transform(X_tv[tr]).astype(np.float32)
    Xvl = scaler.transform(X_tv[vl]).astype(np.float32)
    Xts = scaler.transform(X_te).astype(np.float32)
    tf.keras.utils.set_random_seed(CFG["seed"] + 1000 * si + f)
    model = build_snn_ae(Xtr.shape[1], cfg)
    model.compile(
        optimizer=Adam(cfg["learning_rate"], clipnorm=CFG["grad_clip"]),
        loss="mse",
        metrics=["mae"],
    )
    es = EarlyStopping(
        monitor="val_loss", patience=CFG["patience"], restore_best_weights=True, verbose=0
    )
    t0 = time.time()
    hist = model.fit(
        Xtr,
        Xtr,
        validation_data=(Xvl, Xvl),
        epochs=CFG["epochs"],
        batch_size=CFG["batch_size"],
        callbacks=[es],
        verbose=0,
    )
    pb = CFG["pb"]
    ve = np.mean((Xvl - model.predict(Xvl, batch_size=pb, verbose=0)) ** 2, axis=1)
    teerr = np.mean((Xts - model.predict(Xts, batch_size=pb, verbose=0)) ** 2, axis=1)
    thr = float(np.percentile(ve, CFG["operating_percentile"]))
    yp = (teerr > thr).astype(int)
    sanity = {
        "f1": float(f1_score(y_te, yp)),
        "precision": float(precision_score(y_te, yp, zero_division=0)),
        "recall": float(recall_score(y_te, yp, zero_division=0)),
        "accuracy": float(accuracy_score(y_te, yp)),
        "threshold": thr,
        "train_time_sec": time.time() - t0,
        "feat_std_train_normal": np.std(Xtr, axis=0).tolist(),
    }
    train_idx = tv[tr]
    val_idx = tv[vl]
    split_audit = {
        "dataset": ds,
        "seed_idx": si,
        "fold": f,
        "n_train_normal": int(len(tr)),
        "n_val_normal": int(len(vl)),
        "n_test": int(len(te)),
        "train_attack_count": int(y_eval[train_idx].sum()),
        "val_attack_count": int(y_eval[val_idx].sum()),
        "test_attack_count": int(y_eval[te].sum()),
        "train_val_overlap": int(len(np.intersect1d(train_idx, val_idx))),
        "train_test_overlap": int(len(np.intersect1d(train_idx, te))),
        "val_test_overlap": int(len(np.intersect1d(val_idx, te))),
        "scaler_fit_scope": "training_normal_only",
        "threshold_scope": "validation_normal_only",
        "spike_baseline_scope": "validation_normal_only",
    }
    pd.DataFrame([split_audit]).to_csv(jd / "split_audit.csv", index=False)
    model.save(jd / "model.keras")
    joblib.dump(scaler, jd / "scaler.pkl")
    np.savez_compressed(jd / "indices.npz", tr_local=tr, vl_local=vl, tv=tv, te=te)
    savejson(jd / "threshold.json", {"threshold": thr, "percentile": CFG["operating_percentile"]})
    savejson(jd / "history.json", {k: list(map(float, v)) for k, v in hist.history.items()})
    savejson(jd / "detection_sanity.json", sanity)
    print(f"  [P1][{ds}] s{si}f{f} F1={sanity['f1']:.4f} thr={thr:.4g}")
    del model, Xtr, Xvl, Xts
    tf.keras.backend.clear_session()
    gc.collect()




## Cell 5 — Shared XAI core dan Stage 2 baseline function

In [ ]:
# =============================================================================
# XAI CORE (shared)
# =============================================================================
def rn(a):
    return np.vstack([rankdata(x) / len(x) for x in a]).astype(np.float32)


def err(m, x):
    """Return sample-level MSE and feature-level squared reconstruction error."""
    h = m.predict(x, batch_size=CFG["pb"], verbose=0)
    fe = (x - h) ** 2
    return np.mean(fe, axis=1), fe


def dkl(S, mu):
    a = CFG["alpha"]
    T = 3
    p = (S * T + a) / (T + 2 * a)
    q = (mu[None, :] * T + a) / (T + 2 * a)
    return p * np.log(p / q) + (1 - p) * np.log((1 - p) / (1 - q))


def submodels(model):
    bn = Model(model.input, model.get_layer("batch_norm").output)
    en = Model(model.input, model.get_layer("encoder").output)
    W = model.get_layer("encoder").w.numpy()
    return bn, en, W


# =============================================================================
# STAGE 2 — BASELINE (validation-normal only)
# =============================================================================
def build_baseline(ds, si, f, X_eval, y_eval):
    bd = DIRS["baselines"] / ds / f"seed{si}_fold{f}" / "base"
    bd.mkdir(parents=True, exist_ok=True)
    if (bd / "baseline.npz").exists():
        return
    jd = model_dir(ds, si, f)
    model = tf.keras.models.load_model(jd / "model.keras", custom_objects=CUSTOM)
    scaler = joblib.load(jd / "scaler.pkl")
    idx = np.load(jd / "indices.npz")
    bn, en, W = submodels(model)
    tv, vl = idx["tv"], idx["vl_local"]
    Xvl = scaler.transform(X_eval[tv][vl]).astype(np.float32)
    assert np.all(y_eval[tv][vl] == 0), "LEAK: baseline uses non-normal samples"
    pb = CFG["pb"]
    n, d = Xvl.shape
    SDP = en.predict(Xvl, batch_size=pb, verbose=0).mean(1)
    mu, sd = SDP.mean(0), SDP.std(0)
    _, e = err(model, Xvl)
    err_mu, err_sd = e.mean(0), e.std(0)
    tau_err = err_mu + CFG["sared_k"] * err_sd
    Xbn = bn.predict(Xvl, batch_size=pb, verbose=0)
    dz = np.abs(SDP - mu[None, :]) / (sd[None, :] + 1e-8)
    Dvals = np.empty((n, d), dtype=np.float32)
    for lo in range(0, n, 2048):
        hi = min(n, lo + 2048)
        c = np.abs(Xbn[lo:hi, :, None] * W[None, :, :])
        ch = c / (c.sum(1, keepdims=True) + EPS)
        Dvals[lo:hi] = np.einsum("nji,ni->nj", ch, dz[lo:hi])
    tau_div = np.percentile(Dvals, CFG["sared_pct"], axis=0)
    feat_std = np.array(
        load_json(jd / "detection_sanity.json")["feat_std_train_normal"], dtype=np.float32
    )
    np.savez_compressed(
        bd / "baseline.npz",
        mu_sdp=mu,
        sd_sdp=sd,
        err_mu=err_mu,
        err_sd=err_sd,
        tau_err=tau_err,
        tau_div=tau_div,
        feat_std=feat_std,
        n_val=n,
        Dval_pcts=np.percentile(Dvals, CFG["sared_pct_grid"], axis=0),
        err_for_k=np.stack([err_mu + k * err_sd for k in CFG["sared_k_grid"]]),
    )
    print(f"  [P2][{ds}] s{si}f{f} baseline n_val={n}")
    del model, Xvl, SDP, e, Xbn, Dvals
    tf.keras.backend.clear_session()
    gc.collect()




## Cell 6 — Stage 3 evaluation functions

Cell ini panjang karena berisi semua fungsi evaluasi: faithfulness, KernelSHAP, SARED, stability, suppression, frozen decoder, counterfactual curves, dan per-job export.

In [ ]:
# =============================================================================
# STAGE 3 — EVALUATION (file 4 v3-complete; SHAP full eval preserved)
# =============================================================================
def pool(ds):
    z = np.load(SRC / "data" / f"{ds}_balanced.npz", allow_pickle=True)
    cols = [str(x) for x in z["feature_cols"]]
    assert (
        len(cols) == CFG["dim"]
        and "frame.time_epoch" not in cols
        and "inter_arrival_time" in cols
    ), (ds, cols)
    ei = np.load(V83 / "splits" / f"{ds}_holdout_eval_split.npz")["eval_idx"]
    return z["X"][ei], z["y"][ei], z["label_type"][ei], cols


def job(ds, s, f):
    p = SRC / "models" / ds / f"seed{s}_fold{f}"
    m = tf.keras.models.load_model(p / "model.keras", custom_objects=CUSTOM)
    assert m.input_shape[-1] == CFG["dim"]
    sc = joblib.load(p / "scaler.pkl")
    th = json.load(open(p / "threshold.json"))["threshold"]
    ix = np.load(p / "indices.npz")
    b = np.load(SRC / "baselines" / ds / f"seed{s}_fold{f}" / "base/baseline.npz")
    bn = Model(m.input, m.get_layer("batch_norm").output)
    en = Model(m.input, m.get_layer("encoder").output)
    W = m.get_layer("encoder").w.numpy()
    return m, sc, float(th), ix, b, bn, en, W


def core(m, bn, en, W, b, x):
    S = en.predict(x, batch_size=CFG["pb"], verbose=0).mean(1)
    xb = bn.predict(x, batch_size=CFG["pb"], verbose=0)
    _, e = err(m, x)
    c = np.abs(xb[:, :, None] * W[None, :, :])
    ch = c / (c.sum(1, keepdims=True) + EPS)
    Dz = np.einsum("nji,ni->nj", ch, np.abs(S - b["mu_sdp"]) / (b["sd_sdp"] + 1e-8))
    Dk = np.einsum("nji,ni->nj", ch, dkl(S, b["mu_sdp"]))
    d = x.shape[1]
    re = np.vstack([rankdata(v) / d for v in e])
    return (
        {
            "SDDA_z": rn(Dz),
            "SDDA_KL": rn(Dk),
            "SDDA_RW": rn(Dz * re * d),
            "TSA_adapted": rn(np.einsum("nji,ni->nj", c, S)),
            "RawError": rn(e),
        },
        S,
        Dz,
        e,
    )


def ks(m, x, bg, seed):
    """KernelSHAP attribution on the full selected sample set passed in x.

    The shape handling is intentionally defensive because different SHAP versions
    may return a list, (n, d), (1, n, d), or (n, d, 1) for scalar model outputs.
    No unscored sample is filled with a dummy attribution.
    """
    x = np.asarray(x, np.float32)
    if len(x) == 0:
        raise RuntimeError("KernelSHAP received an empty sample set")
    np.random.seed(seed)

    def fn(z):
        return err(m, np.asarray(z, np.float32))[0]

    sv = shap.KernelExplainer(fn, bg).shap_values(
        x, nsamples=CFG["shap_nsamples"], silent=True, seed=seed
    )
    if isinstance(sv, list):
        if len(sv) != 1:
            raise RuntimeError(f"Unexpected SHAP list length: {len(sv)}")
        sv = sv[0]
    v = np.asarray(sv)
    if v.ndim == 3 and v.shape[0] == 1 and v.shape[1:] == x.shape:
        v = v[0]
    elif v.ndim == 3 and v.shape[:2] == x.shape and v.shape[2] == 1:
        v = v[:, :, 0]
    elif v.ndim == 1 and x.shape[0] == 1 and v.shape[0] == x.shape[1]:
        v = v.reshape(1, -1)
    if v.shape != x.shape:
        raise RuntimeError(f"SHAP shape {v.shape} != {x.shape}")
    return rn(np.abs(v))


def scores(m, bn, en, W, b, x, bg, seed):
    q, S, Dz, e = core(m, bn, en, W, b, x)
    q["KernelSHAP"] = ks(m, x, bg, seed)
    q["Random"] = np.random.RandomState(seed + 17).random(x.shape).astype(np.float32)
    return q, S, Dz, e


def one(method, m, bn, en, W, b, x, bg, seed):
    if method == "KernelSHAP":
        return ks(m, x, bg, seed)
    if method == "Random":
        return np.random.RandomState(seed + 17).random(x.shape)
    if method == "RawError":
        return rn(err(m, x)[1])
    S = en.predict(x, batch_size=CFG["pb"], verbose=0).mean(1)
    xb = bn.predict(x, batch_size=CFG["pb"], verbose=0)
    c = np.abs(xb[:, :, None] * W[None, :, :])
    if method == "TSA_adapted":
        return rn(np.einsum("nji,ni->nj", c, S))
    ch = c / (c.sum(1, keepdims=True) + EPS)
    div = dkl(S, b["mu_sdp"]) if method == "SDDA_KL" else np.abs(S - b["mu_sdp"]) / (b["sd_sdp"] + 1e-8)
    D = np.einsum("nji,ni->nj", ch, div)
    if method != "SDDA_RW":
        return rn(D)
    e = err(m, x)[1]
    d = x.shape[1]
    re = np.vstack([rankdata(v) / d for v in e])
    return rn(D * re * d)


def supp(S, mu):
    M = dkl(S, mu)
    sign = np.sign(S - mu[None, :])
    a = np.where(sign < 0, M, 0).mean(1)
    b = np.where(sign > 0, M, 0).mean(1)
    return a / (a + b + EPS)


def sared(m, bn, en, W, xv, xa, seed):
    r = np.random.RandomState(seed).permutation(len(xv))
    cut = len(r) // 2
    xc, xe = xv[r[:cut]], xv[r[cut:]]

    def arrays(x, mu=None, sd=None):
        S = en.predict(x, batch_size=CFG["pb"], verbose=0).mean(1)
        xb = bn.predict(x, batch_size=CFG["pb"], verbose=0)
        _, e = err(m, x)
        if mu is None:
            mu, sd = S.mean(0), S.std(0)
        c = np.abs(xb[:, :, None] * W[None, :, :])
        c /= c.sum(1, keepdims=True) + EPS
        D = np.einsum("nji,ni->nj", c, np.abs(S - mu) / (sd + 1e-8))
        return e, D, mu, sd

    ec, Dc, mu, sd = arrays(xc)
    te = ec.mean(0) + CFG["sared_k"] * ec.std(0)
    td = np.percentile(Dc, CFG["sared_pct"], axis=0)

    def classify(x):
        e, D, _, _ = arrays(x, mu, sd)
        hi = e > te
        y = np.zeros(e.shape, np.int8)
        y[hi & (D > td)] = 1
        y[hi & (D <= td)] = 2
        return y

    ne, aa = classify(xe), classify(xa)
    return {
        "n_cal": len(xc),
        "n_eval": len(xe),
        "normal_false_flag": float((ne > 0).mean()),
        "normal_encoding": float((ne == 1).mean()),
        "normal_decoding": float((ne == 2).mean()),
        "anomaly_normal": float((aa == 0).mean()),
        "anomaly_encoding": float((aa == 1).mean()),
        "anomaly_decoding": float((aa == 2).mean()),
    }, aa


def ranking_metrics(a, truth):
    order = np.argsort(-a)
    pos = np.empty(len(a), int)
    pos[order] = np.arange(1, len(a) + 1)
    y = np.zeros(len(a), int)
    y[list(truth)] = 1
    k = min(5, len(a))
    dcg = sum(y[order[i]] / np.log2(i + 2) for i in range(k))
    ideal = sum(1 / np.log2(i + 2) for i in range(min(k, len(truth))))
    return {
        "p1": len(set(order[:1]) & set(truth)),
        "p3": len(set(order[:3]) & set(truth)) / 3,
        "r5": len(set(order[:5]) & set(truth)) / len(truth),
        "mean_rank": np.mean([pos[j] for j in truth]),
        "mrr": 1.0 / min(pos[j] for j in truth),
        "ndcg5": dcg / ideal,
        "average_precision": average_precision_score(y, a),
        "feature_auc": roc_auc_score(y, a),
    }


def stability(m, bn, en, W, b, xa, bg, feat_std, seed):
    rng0 = np.random.RandomState(seed)
    n = min(CFG["stability_n"], len(xa))
    x = xa[rng0.choice(len(xa), n, replace=False)]
    rows = []

    def emit(method, regime, ep, rep, a, bb):
        for i, (u, v) in enumerate(zip(a, bb)):
            sp = spearmanr(u, v).statistic
            A, B = set(np.argsort(-u)[:5]), set(np.argsort(-v)[:5])
            rows.append(
                {
                    "method": method,
                    "regime": regime,
                    "eps": ep,
                    "repeat": rep,
                    "sample": i,
                    "spearman": sp,
                    "jaccard5": len(A & B) / len(A | B),
                    "n": len(x),
                }
            )

    for method in METHODS:
        reps = CFG["shap_self_repeats"] if method in ["KernelSHAP", "Random"] else 2
        runs = [one(method, m, bn, en, W, b, x, bg, seed + 100 + i) for i in range(reps)]
        rep = 0
        for i in range(reps):
            for j in range(i + 1, reps):
                emit(method, "self", 0.0, rep, runs[i], runs[j])
                rep += 1
    for ep in CFG["stability_eps"]:
        base = {method: one(method, m, bn, en, W, b, x, bg, seed + 500) for method in METHODS}
        rng = np.random.RandomState(seed + int(ep * 10000))
        for rep in range(CFG["stability_repeats"]):
            xp = x + rng.normal(size=x.shape) * (ep * feat_std)[None, :]
            for method in METHODS:
                method_seed = seed + 500 + (rep + 1 if method == "Random" else 0)
                emit(
                    method,
                    "input_noise",
                    ep,
                    rep,
                    base[method],
                    one(method, m, bn, en, W, b, xp, bg, method_seed),
                )
    return rows


def sensitivity(m, bn, en, W, xv, seed):
    rng = np.random.RandomState(seed)
    ix = rng.permutation(len(xv))
    cut = len(ix) // 2
    xc, xe = xv[ix[:cut]], xv[ix[cut:]]

    def vals(x, mu=None, sd=None):
        S = en.predict(x, batch_size=CFG["pb"], verbose=0).mean(1)
        xb = bn.predict(x, batch_size=CFG["pb"], verbose=0)
        _, e = err(m, x)
        if mu is None:
            mu, sd = S.mean(0), S.std(0)
        c = np.abs(xb[:, :, None] * W[None, :, :])
        c /= c.sum(1, keepdims=True) + EPS
        Dz = np.einsum("nji,ni->nj", c, np.abs(S - mu) / (sd + 1e-8))
        return S, e, Dz, c, mu, sd

    Sc, ec, Dc, cc, mu, sd = vals(xc)
    Se, ee, De, ce, _, _ = vals(xe, mu, sd)
    rows = []
    for k in CFG["sared_k_grid"]:
        te = ec.mean(0) + k * ec.std(0)
        for pct in CFG["sared_pct_grid"]:
            td = np.percentile(Dc, pct, axis=0)
            high = ee > te
            cat = np.zeros(ee.shape, np.int8)
            cat[high & (De > td)] = 1
            cat[high & (De <= td)] = 2
            rows.append(
                {
                    "kind": "sared_threshold",
                    "k": k,
                    "div_pct": pct,
                    "alpha": np.nan,
                    "value": float((cat > 0).mean()),
                    "encoding": float((cat == 1).mean()),
                    "decoding": float((cat == 2).mean()),
                    "n": len(xe),
                }
            )
    for a in CFG["alpha_grid"]:
        old = CFG["alpha"]
        CFG["alpha"] = a
        Dk = np.einsum("nji,ni->nj", ce, dkl(Se, mu))
        CFG["alpha"] = old
        agreement = np.nanmean([spearmanr(De[i], Dk[i]).statistic for i in range(len(xe))])
        rows.append(
            {
                "kind": "alpha_agreement",
                "k": np.nan,
                "div_pct": np.nan,
                "alpha": a,
                "value": float(agreement),
                "encoding": np.nan,
                "decoding": np.nan,
                "n": len(xe),
            }
        )
    return rows


def suppression_full(m, en, b, xt, yt, lt, det, rng):
    rows = []
    for st in np.unique(lt[yt == 1]):
        cand = np.where(det & (lt == st))[0]
        if len(cand) > CFG["supp_per_subtype"]:
            cand = rng.choice(cand, CFG["supp_per_subtype"], replace=False)
        if not len(cand):
            continue
        S = en.predict(xt[cand], batch_size=CFG["pb"], verbose=0).mean(1)
        z = (S - b["mu_sdp"]) / (b["sd_sdp"] + 1e-8)
        M = dkl(S, b["mu_sdp"])
        for kind, mag in [("continuous_z", np.abs(z)), ("bernoulli_kl", M)]:
            direction = np.sign(S - b["mu_sdp"][None, :])
            su = np.where(direction < 0, mag, 0).mean(1)
            ex = np.where(direction > 0, mag, 0).mean(1)
            ratio = su / (su + ex + EPS)
            for i, idx in enumerate(cand):
                rows.append(
                    {
                        "subtype": str(st),
                        "metric": kind,
                        "sample_test_local_idx": int(idx),
                        "n": 1,
                        "suppression_score": float(su[i]),
                        "excitation_score": float(ex[i]),
                        "suppression_ratio": float(ratio[i]),
                    }
                )
    return rows


def decoder_head(width, input_shape, d, decoder_cfg, seed):
    tf.keras.utils.set_random_seed(seed)
    inp = Input(shape=input_shape)
    x = SpikingLayer(
        width,
        threshold=decoder_cfg["threshold"],
        decay=decoder_cfg["decay"],
        surrogate_scale=decoder_cfg["surrogate_scale"],
        time_steps=decoder_cfg["time_steps"],
        return_sequences=False,
        name=f"decoder_{width}",
    )(inp)
    out = Dense(d, activation="linear", name=f"output_{width}")(x)
    h = Model(inp, out)
    h.compile(optimizer=Adam(CFG["decoder_learning_rate"]), loss="mse")
    return h


def frozen_decoder(m, bn, en, W, b, xtrain, xv, xa, seed, outdir):
    rng = np.random.RandomState(seed)
    if len(xtrain) > CFG["decoder_train_n"]:
        xtrain = xtrain[rng.choice(len(xtrain), CFG["decoder_train_n"], replace=False)]
    ztr = en.predict(xtrain, batch_size=CFG["pb"], verbose=0)
    zv = en.predict(xv, batch_size=CFG["pb"], verbose=0)
    za = en.predict(xa, batch_size=CFG["pb"], verbose=0)
    dc = m.get_layer("decoder").get_config()
    heads, histories = {}, []
    for width in CFG["decoder_widths"]:
        h = decoder_head(width, ztr.shape[1:], xtrain.shape[1], dc, seed + width)
        hist = h.fit(
            ztr,
            xtrain,
            validation_data=(zv, xv),
            epochs=CFG["decoder_epochs"],
            batch_size=CFG["decoder_batch"],
            callbacks=[
                EarlyStopping(
                    monitor="val_loss", patience=CFG["decoder_patience"], restore_best_weights=True
                )
            ],
            verbose=0,
        )
        h.save(Path(outdir) / f"frozen_decoder_{width}.keras")
        heads[width] = h
        histories.append(
            {"width": width, "epochs": len(hist.history["loss"]), "best_val_loss": min(hist.history["val_loss"])}
        )
    Sval, Sano = zv.mean(1), za.mean(1)
    xbval = bn.predict(xv, batch_size=CFG["pb"], verbose=0)
    xbano = bn.predict(xa, batch_size=CFG["pb"], verbose=0)

    def D(xb, S):
        c = np.abs(xb[:, :, None] * W[None, :, :])
        c /= c.sum(1, keepdims=True) + EPS
        return np.einsum("nji,ni->nj", c, np.abs(S - b["mu_sdp"]) / (b["sd_sdp"] + 1e-8))

    Dv, Da = D(xbval, Sval), D(xbano, Sano)
    td = np.percentile(Dv, CFG["sared_pct"], axis=0)
    base = CFG["decoder_widths"][0]
    evals = {w: (xv - heads[w].predict(zv, batch_size=CFG["pb"], verbose=0)) ** 2 for w in heads}
    te = evals[base].mean(0) + CFG["sared_k"] * evals[base].std(0)
    cats = {}
    for w, h in heads.items():
        e = (xa - h.predict(za, batch_size=CFG["pb"], verbose=0)) ** 2
        high = e > te
        c = np.zeros(e.shape, np.int8)
        c[high & (Da > td)] = 1
        c[high & (Da <= td)] = 2
        cats[w] = c
    names = ["NORMAL", "ENCODING_FAILURE", "DECODING_LIMITATION"]
    rows = []
    c0, c1 = cats[base], cats[CFG["decoder_widths"][1]]
    for a in range(3):
        for bb in range(3):
            count = int(((c0 == a) & (c1 == bb)).sum())
            den = max(1, int((c0 == a).sum()))
            rows.append({"from": names[a], "to": names[bb], "count": count, "pct_of_from": count / den})
    return rows, histories, c0, c1


def pick_sub(y, lab, det, n, rng):
    pick, cov = [], []
    for st in np.unique(lab[y == 1]):
        eligible = np.where((y == 1) & (lab == st))[0]
        cand = np.where(det & (lab == st))[0]
        take = min(n, len(cand))
        sel = rng.choice(cand, take, replace=False) if take else []
        pick.extend(sel)
        cov.append({"subtype": str(st), "eligible": len(eligible), "detected": len(cand), "selected": take})
    return np.asarray(pick, int), cov


def curves(m, x, order, ref, th):
    n, d = x.shape
    de = x.copy()
    ins = ref.copy()
    A = np.zeros((n, d + 1), np.float32)
    B = A.copy()
    A[:, 0] = err(m, de)[0]
    B[:, 0] = err(m, ins)[0]
    kd = np.full(n, np.nan)
    ki = np.full(n, np.nan)
    ki[B[:, 0] > th] = 0
    for k in range(d):
        rr = np.arange(n)
        cc = order[:, k]
        de[rr, cc] = ref[rr, cc]
        ins[rr, cc] = x[rr, cc]
        A[:, k + 1] = err(m, de)[0]
        B[:, k + 1] = err(m, ins)[0]
        z = np.isnan(kd) & (A[:, k + 1] <= th)
        kd[z] = k + 1
        z = np.isnan(ki) & (B[:, k + 1] > th)
        ki[z] = k + 1
    return kd, ki, A, B


def run(ds, s, f, X, y, lab, cols):
    od = OUT / "jobs" / f"{ds}_seed{s}_fold{f}"
    od.mkdir(parents=True, exist_ok=True)
    done = od / "done.json"
    if done.exists() and json.load(open(done)).get("run_hash") == RH:
        print("[skip]", ds, s, f)
        return
    rng = np.random.RandomState(CFG["seed"] + 1000 * s + f)
    m, sc, th, ix, b, bn, en, W = job(ds, s, f)
    tv, te, vl, tr = ix["tv"], ix["te"], ix["vl_local"], ix["tr_local"]
    xt = sc.transform(X[te]).astype(np.float32)
    yt, lt = y[te], lab[te]
    xv = sc.transform(X[tv[vl]]).astype(np.float32)
    assert np.all(y[tv[vl]] == 0)
    ae, _ = err(m, xt)
    yp = (ae > th).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
    pd.DataFrame(
        [
            {
                "dataset": ds,
                "seed_idx": s,
                "fold": f,
                "accuracy": accuracy_score(yt, yp),
                "precision": precision_score(yt, yp, zero_division=0),
                "recall": recall_score(yt, yp, zero_division=0),
                "f1": f1_score(yt, yp, zero_division=0),
                "roc_auc": roc_auc_score(yt, ae),
                "pr_auc": average_precision_score(yt, ae),
                "threshold": th,
                "tn": tn,
                "fp": fp,
                "fn": fn,
                "tp": tp,
                "n_test": len(yt),
                "prevalence": yt.mean(),
            }
        ]
    ).to_csv(od / "detection.csv", index=False)
    np.savez_compressed(
        od / "detection_scores.npz",
        y_true=yt.astype(np.int8),
        score=ae.astype(np.float32),
        pred=yp.astype(np.int8),
        eval_local_idx=te,
    )
    bg = xv[rng.choice(len(xv), min(CFG["shap_bg"], len(xv)), replace=False)]

    src = np.where((yt == 0) & (ae <= th))[0]
    n = min(CFG["faith_n"], len(src))
    if n <= 0:
        raise RuntimeError("No correctly classified normal samples available for controlled faithfulness")
    fpick = rng.choice(src, n, replace=False)
    sd = np.asarray(b["feat_std"])
    valid = np.where(sd > 1e-8)[0]
    if len(valid) < CFG["m_features"]:
        raise RuntimeError(f"Only {len(valid)} perturbable features, need {CFG['m_features']}")
    truth = [rng.choice(valid, CFG["m_features"], replace=False) for _ in range(n)]
    signs = rng.choice([-1.0, 1.0], (n, CFG["m_features"]))
    fr, rt, peff = [], [], []
    for mag in CFG["magnitudes"]:
        xp = xt[fpick].copy()
        for i, fs in enumerate(truth):
            xp[i, fs] += signs[i] * mag * sd[fs]
        changed_slots = int(sum(np.sum(xp[i, fs] != xt[fpick[i], fs]) for i, fs in enumerate(truth)))
        selected_slots = int(n * CFG["m_features"])
        assert changed_slots == selected_slots
        peff.append({
            "dataset": ds,
            "seed_idx": s,
            "fold": f,
            "perturbation_mode": "scaled_std",
            "magnitude": mag,
            "selected_slots": selected_slots,
            "changed_slots": changed_slots,
            "unchanged_slots": selected_slots - changed_slots,
            "zero_std_selected": 0,
            "n_samples": int(n),
        })
        Q, _, _, _ = scores(m, bn, en, W, b, xp, bg, 7000 + 100 * s + f)
        pe = err(m, xp)[0]
        for method, A in Q.items():
            for i, fs in enumerate(truth):
                met = ranking_metrics(A[i], set(map(int, fs)))
                fr.append(
                    {
                        "dataset": ds,
                        "seed_idx": s,
                        "fold": f,
                        "method": method,
                        "perturbation_mode": "scaled_std",
                        "magnitude": mag,
                        "sample": i,
                        "test_local_idx": int(fpick[i]),
                        "eval_local_idx": int(te[fpick[i]]),
                        "detected": pe[i] > th,
                        **met,
                    }
                )
        np.savez_compressed(
            od / f"faith_mag{mag}.npz",
            truth=np.asarray(truth),
            test_local_idx=fpick,
            eval_local_idx=te[fpick],
            perturbed_error=pe,
            **Q,
        )
        if mag == max(CFG["magnitudes"]):
            xr = xp[: min(CFG["runtime_n"], len(xp))]
            for method in METHODS:
                reps = 1 if method == "KernelSHAP" else CFG["runtime_repeats"]
                one(method, m, bn, en, W, b, xr, bg, 8100 + s * 100 + f)
                for rep in range(reps):
                    t0 = time.perf_counter()
                    one(method, m, bn, en, W, b, xr, bg, 8200 + s * 100 + f + rep)
                    rt.append(
                        {
                            "dataset": ds,
                            "seed_idx": s,
                            "fold": f,
                            "method": method,
                            "repeat": rep,
                            "n": len(xr),
                            "seconds_per_sample": (time.perf_counter() - t0) / len(xr),
                        }
                    )
    donor_pool = np.where(y[tv] == 1)[0]
    donor_idx = rng.choice(donor_pool, n, replace=len(donor_pool) < n)
    donors = sc.transform(X[tv[donor_idx]]).astype(np.float32)
    xd = xt[fpick].copy()
    donor_truth = []
    for i in range(n):
        candidate = valid[np.abs(donors[i, valid] - xd[i, valid]) > 1e-8]
        if len(candidate) < CFG["m_features"]:
            raise RuntimeError("Insufficient changed features in empirical donor perturbation")
        fs = rng.choice(candidate, CFG["m_features"], replace=False)
        donor_truth.append(fs)
        xd[i, fs] = donors[i, fs]
    donor_selected_slots = int(n * CFG["m_features"])
    donor_changed_slots = int(sum(np.sum(xd[i, fs] != xt[fpick[i], fs]) for i, fs in enumerate(donor_truth)))
    assert donor_changed_slots == donor_selected_slots
    peff.append({
        "dataset": ds,
        "seed_idx": s,
        "fold": f,
        "perturbation_mode": "empirical_attack_donor",
        "magnitude": 1.0,
        "selected_slots": donor_selected_slots,
        "changed_slots": donor_changed_slots,
        "unchanged_slots": donor_selected_slots - donor_changed_slots,
        "zero_std_selected": 0,
        "n_samples": int(n),
    })
    Qd, _, _, _ = scores(m, bn, en, W, b, xd, bg, 7500 + 100 * s + f)
    ped = err(m, xd)[0]
    for method, A in Qd.items():
        for i, fs in enumerate(donor_truth):
            met = ranking_metrics(A[i], set(map(int, fs)))
            fr.append(
                {
                    "dataset": ds,
                    "seed_idx": s,
                    "fold": f,
                    "method": method,
                    "perturbation_mode": "empirical_attack_donor",
                    "magnitude": 1.0,
                    "sample": i,
                    "test_local_idx": int(fpick[i]),
                    "eval_local_idx": int(te[fpick[i]]),
                    "detected": ped[i] > th,
                    **met,
                }
            )
    np.savez_compressed(
        od / "faith_empirical_donor.npz",
        truth=np.asarray(donor_truth),
        donor_trainval_local_idx=donor_idx,
        test_local_idx=fpick,
        eval_local_idx=te[fpick],
        perturbed_error=ped,
        **Qd,
    )
    pd.DataFrame(fr).to_csv(od / "faithfulness.csv", index=False)
    pd.DataFrame(rt).to_csv(od / "runtime.csv", index=False)
    pd.DataFrame(peff).to_csv(od / "perturbation_effectiveness.csv", index=False)

    det = (yt == 1) & (ae > th)
    cpick, cov = pick_sub(yt, lt, det, CFG["cf_per_subtype"], rng)
    for z in cov:
        z.update({"dataset": ds, "seed_idx": s, "fold": f})
    pd.DataFrame(cov).to_csv(od / "coverage.csv", index=False)
    if not len(cpick):
        raise RuntimeError("No detected anomalies")
    xa, la = xt[cpick], lt[cpick]
    ve = err(m, xv)[0]
    refs = np.where(ve <= th)[0]
    if len(refs) > CFG["reference_pool"]:
        refs = rng.choice(refs, CFG["reference_pool"], replace=False)
    poolref = xv[refs]
    nn = NearestNeighbors(n_neighbors=1).fit(poolref)
    ref = poolref[nn.kneighbors(xa, return_distance=False)[:, 0]]
    assert np.all(err(m, ref)[0] <= th)
    Q, S, _, _ = scores(m, bn, en, W, b, xa, bg, 9000 + s * 100 + f)
    atk = np.where(y[tv] == 1)[0]
    if len(atk) > CFG["supp_cal_n"]:
        atk = rng.choice(atk, CFG["supp_cal_n"], replace=False)
    Sac = en.predict(sc.transform(X[tv[atk]]).astype(np.float32), batch_size=CFG["pb"], verbose=0).mean(1)
    st = float(np.median(supp(Sac, b["mu_sdp"])))
    sr = supp(S, b["mu_sdp"])
    strata = np.where(sr >= st, "high", "low")
    ss, cat = sared(m, bn, en, W, xv, xa, 5000 + 100 * s + f)
    ss.update({"dataset": ds, "seed_idx": s, "fold": f, "n_anomaly": len(xa)})
    pd.DataFrame([ss]).to_csv(od / "sared.csv", index=False)
    np.savez_compressed(
        od / "real_anomaly_attributions.npz",
        test_local_idx=cpick,
        eval_local_idx=te[cpick],
        subtype=la,
        sared=cat,
        **Q,
    )
    top = []
    for subtype in np.unique(la):
        mask = la == subtype
        for method, A in Q.items():
            avg = A[mask].mean(0)
            ix5 = np.argsort(-avg)[:5]
            top.append(
                {
                    "dataset": ds,
                    "seed_idx": s,
                    "fold": f,
                    "subtype": str(subtype),
                    "method": method,
                    "n": int(mask.sum()),
                    "top5_features": ";".join(map(str, ix5)),
                    "top5_feature_names": ";".join(cols[j] for j in ix5),
                    "top5_scores": ";".join(f"{avg[j]:.8g}" for j in ix5),
                }
            )
    pd.DataFrame(top).to_csv(od / "per_attack_topk.csv", index=False)
    local = []
    cat_names = {0: "Normal", 1: "Encoding Failure", 2: "Decoding Limitation"}
    for method, A in Q.items():
        for i in range(len(xa)):
            ix5 = np.argsort(-A[i])[:5]
            local.append(
                {
                    "dataset": ds,
                    "seed_idx": s,
                    "fold": f,
                    "method": method,
                    "sample": i,
                    "test_local_idx": int(cpick[i]),
                    "eval_local_idx": int(te[cpick[i]]),
                    "subtype": str(la[i]),
                    "top5_features": ";".join(map(str, ix5)),
                    "top5_feature_names": ";".join(cols[j] for j in ix5),
                    "top5_scores": ";".join(f"{A[i,j]:.8g}" for j in ix5),
                    "top5_sared_categories": ";".join(cat_names[int(cat[i, j])] for j in ix5),
                }
            )
    pd.DataFrame(local).to_csv(od / "local_explanations.csv", index=False)
    normal_raw = X[tv[vl]]
    normal_sel = rng.choice(
        len(normal_raw), min(CFG["feature_behavior_normal_n"], len(normal_raw)), replace=False
    )
    behavior = []
    for subtype in np.unique(la):
        mask = la == subtype
        candidates = np.setdiff1d(np.where(det & (lt == subtype))[0], cpick, assume_unique=False)
        independent = len(candidates) > 0
        if not independent:
            candidates = cpick[mask]
        if len(candidates) > CFG["feature_behavior_normal_n"]:
            candidates = rng.choice(candidates, CFG["feature_behavior_normal_n"], replace=False)
        attack_values = X[te[candidates]]
        for method, A in Q.items():
            avg = A[mask].mean(0)
            topidx = np.argsort(-avg)[: CFG["feature_behavior_topk"]]
            for j in topidx:
                for value in normal_raw[normal_sel, j]:
                    behavior.append(
                        {
                            "dataset": ds,
                            "seed_idx": s,
                            "fold": f,
                            "method": method,
                            "subtype": str(subtype),
                            "feature_idx": int(j),
                            "feature_name": cols[j],
                            "group": "normal",
                            "value": float(value),
                            "independent_from_explanation_sample": True,
                        }
                    )
                for value in attack_values[:, j]:
                    behavior.append(
                        {
                            "dataset": ds,
                            "seed_idx": s,
                            "fold": f,
                            "method": method,
                            "subtype": str(subtype),
                            "feature_idx": int(j),
                            "feature_name": cols[j],
                            "group": "attack",
                            "value": float(value),
                            "independent_from_explanation_sample": independent,
                        }
                    )
    pd.DataFrame(behavior).to_csv(od / "feature_behavior.csv", index=False)
    agree = []
    for a, bm in combinations(METHODS, 2):
        vals = [spearmanr(Q[a][i], Q[bm][i]).statistic for i in range(len(xa))]
        agree.append(
            {"dataset": ds, "seed_idx": s, "fold": f, "pair": f"{a} vs {bm}", "spearman_mean": float(np.nanmean(vals)), "n": len(vals)}
        )
    pd.DataFrame(agree).to_csv(od / "method_agreement.csv", index=False)
    sdist = []
    for subtype in np.unique(la):
        c = cat[la == subtype]
        sdist.append(
            {
                "dataset": ds,
                "seed_idx": s,
                "fold": f,
                "subtype": str(subtype),
                "n_samples": int((la == subtype).sum()),
                "pct_normal": float((c == 0).mean()),
                "pct_encoding_failure": float((c == 1).mean()),
                "pct_decoding_limitation": float((c == 2).mean()),
            }
        )
    pd.DataFrame(sdist).to_csv(od / "sared_distribution.csv", index=False)
    stab = stability(m, bn, en, W, b, xa, bg, sd, 10000 + 100 * s + f)
    for z in stab:
        z.update({"dataset": ds, "seed_idx": s, "fold": f})
    pd.DataFrame(stab).to_csv(od / "stability.csv", index=False)
    sens = sensitivity(m, bn, en, W, xv, 11000 + 100 * s + f)
    for z in sens:
        z.update({"dataset": ds, "seed_idx": s, "fold": f})
    pd.DataFrame(sens).to_csv(od / "sensitivity.csv", index=False)
    su = suppression_full(m, en, b, xt, yt, lt, det, rng)
    for z in su:
        z.update({"dataset": ds, "seed_idx": s, "fold": f})
    pd.DataFrame(su).to_csv(od / "suppression.csv", index=False)
    if CFG["run_frozen_decoder_intervention"]:
        xtrain = sc.transform(X[tv[tr]]).astype(np.float32)
        trans, hist, cbase, cwide = frozen_decoder(m, bn, en, W, b, xtrain, xv, xa, 12000 + 100 * s + f, od)
        np.savez_compressed(
            od / "frozen_decoder_categories.npz",
            test_local_idx=cpick,
            eval_local_idx=te[cpick],
            subtype=la,
            base_category=cbase,
            wide_category=cwide,
        )
        for z in trans:
            z.update({"dataset": ds, "seed_idx": s, "fold": f})
        for z in hist:
            z.update({"dataset": ds, "seed_idx": s, "fold": f})
        pd.DataFrame(trans).to_csv(od / "decoder_transition.csv", index=False)
        pd.DataFrame(hist).to_csv(od / "decoder_training.csv", index=False)
        del xtrain
    cr, carr = [], {}
    for method, A in Q.items():
        kd, ki, de, ins = curves(m, xa, np.argsort(-A, axis=1), ref, th)
        carr[method + "_deletion"] = de
        carr[method + "_insertion"] = ins
        # AUDC/AUIC are computed as normalized trapezoidal areas over the log-normalized curves.
        # Lower log_audc indicates faster anomaly reduction under deletion; higher/faster log_auic
        # indicates stronger anomaly restoration under insertion.
        log_de = np.log1p(de / th)
        log_ins = np.log1p(ins / th)
        ad = np.trapz(log_de, axis=1) / (de.shape[1] - 1)
        ai = np.trapz(log_ins, axis=1) / (ins.shape[1] - 1)
        for i in range(len(xa)):
            cr.append(
                {
                    "dataset": ds,
                    "seed_idx": s,
                    "fold": f,
                    "method": method,
                    "sample": i,
                    "test_local_idx": int(cpick[i]),
                    "eval_local_idx": int(te[cpick[i]]),
                    "subtype": str(la[i]),
                    "suppression_ratio": sr[i],
                    "suppression_threshold": st,
                    "stratum": strata[i],
                    "sared_encoding_features": int((cat[i] == 1).sum()),
                    "sared_decoding_features": int((cat[i] == 2).sum()),
                    "deletion_flip": np.isfinite(kd[i]),
                    "deletion_k": kd[i],
                    "restricted_deletion_k": (kd[i] if np.isfinite(kd[i]) else xa.shape[1]),
                    "insertion_flip": np.isfinite(ki[i]),
                    "insertion_k": ki[i],
                    "restricted_insertion_k": (ki[i] if np.isfinite(ki[i]) else xa.shape[1]),
                    "log_audc": ad[i],
                    "log_auic": ai[i],
                    "max_k": xa.shape[1],
                }
            )
    pd.DataFrame(cr).to_csv(od / "counterfactual.csv", index=False)
    np.savez_compressed(
        od / "counterfactual_curves.npz",
        test_local_idx=cpick,
        eval_local_idx=te[cpick],
        subtype=la,
        reference=ref,
        suppression_ratio=sr,
        stratum=strata,
        **carr,
    )
    savejson(done, {"run_hash": RH, "completed": time.strftime("%F %T")})
    print("[done]", ds, s, f, "faith", n, "cf", len(cpick))
    del m, xt, xv, xa, Q
    tf.keras.backend.clear_session()
    gc.collect()




## Cell 7 — Export/statistics functions

In [ ]:
# =============================================================================
# EXPORT + statistics (all-pairs Wilcoxon+Holm over 11 metrics)
# =============================================================================
def combine(name):
    p = sorted((OUT / "jobs").glob(f"*/{name}"))
    return pd.concat([pd.read_csv(x) for x in p], ignore_index=True) if p else pd.DataFrame()


def holm(p):
    out, prev = {}, 0.0
    a = sorted(p.items(), key=lambda x: x[1])
    n = len(a)
    for i, (k, v) in enumerate(a):
        prev = max(prev, (n - i) * v)
        out[k] = min(1.0, prev)
    return out


def ci(v, seed):
    v = np.asarray(v, float)
    r = np.random.RandomState(seed)
    q = [r.choice(v, len(v), replace=True).mean() for _ in range(CFG["bootstrap"])]
    return np.percentile(q, [2.5, 97.5])


def all_pairs(run_level, metric, seed0):
    """Wilcoxon + Holm + bootstrap CI, paired on (seed_idx,fold), per dataset."""
    stats = []
    for ds, g in run_level.groupby("dataset"):
        P = g.pivot(index=["seed_idx", "fold"], columns="method", values="value")
        ms = [m for m in METHODS if m in P and P[m].notna().all()]
        raw, info = {}, {}
        for a, bm in combinations(ms, 2):
            d = (P[a] - P[bm]).values
            k = f"{a} vs {bm}"
            try:
                raw[k] = float(wilcoxon(d).pvalue)
            except Exception:
                raw[k] = 1.0
            lo, hi = ci(d, seed0 + len(stats))
            info[k] = (float(d.mean()), lo, hi, len(d))
        adj = holm(raw)
        for k in raw:
            md, lo, hi, n = info[k]
            stats.append(
                {
                    "dataset": ds,
                    "metric": metric,
                    "comparison": k,
                    "p_raw": raw[k],
                    "p_holm_all_pairs": adj[k],
                    "mean_difference": md,
                    "ci95_low": lo,
                    "ci95_high": hi,
                    "n_run_pairs": n,
                }
            )
    return stats



def combine_model_files(name):
    p = sorted((OUT / "models").glob(f"*/*/{name}"))
    return pd.concat([pd.read_csv(x) for x in p], ignore_index=True) if p else pd.DataFrame()

def export():
    names = {
        "detection": "detection.csv",
        "faithfulness": "faithfulness.csv",
        "perturbation_effectiveness": "perturbation_effectiveness.csv",
        "runtime": "runtime.csv",
        "counterfactual": "counterfactual.csv",
        "coverage": "coverage.csv",
        "sared": "sared.csv",
        "topk": "per_attack_topk.csv",
        "local": "local_explanations.csv",
        "feature_behavior": "feature_behavior.csv",
        "agreement": "method_agreement.csv",
        "sared_distribution": "sared_distribution.csv",
        "stability": "stability.csv",
        "sensitivity": "sensitivity.csv",
        "suppression": "suppression.csv",
        "decoder_transition": "decoder_transition.csv",
        "decoder_training": "decoder_training.csv",
    }
    Z = {k: combine(v) for k, v in names.items()}
    for name, df in Z.items():
        if not df.empty:
            df.to_csv(OUT / "raw" / f"{name}.csv", index=False)
    split_audit = combine_model_files("split_audit.csv")
    if not split_audit.empty:
        split_audit.to_csv(OUT / "raw/split_audit.csv", index=False)
        split_audit.to_csv(OUT / "tables/table_split_audit.csv", index=False)
    if not Z["perturbation_effectiveness"].empty:
        Z["perturbation_effectiveness"].to_csv(OUT / "tables/table_perturbation_effectiveness.csv", index=False)
    D, F, R, C, V, S = Z["detection"], Z["faithfulness"], Z["runtime"], Z["counterfactual"], Z["coverage"], Z["sared"]
    D.to_csv(OUT / "tables/table_detection_raw.csv", index=False)
    D.groupby("dataset").agg(
        **{m + "_mean": (m, "mean") for m in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]},
        **{m + "_std": (m, "std") for m in ["f1", "roc_auc", "pr_auc"]},
        n_runs=("fold", "count"),
    ).reset_index().to_csv(OUT / "tables/table_detection.csv", index=False)
    Fr = (
        F.groupby(["dataset", "seed_idx", "fold", "method", "perturbation_mode", "magnitude"])
        .agg(
            p1=("p1", "mean"),
            p3=("p3", "mean"),
            r5=("r5", "mean"),
            mean_rank=("mean_rank", "mean"),
            mrr=("mrr", "mean"),
            ndcg5=("ndcg5", "mean"),
            average_precision=("average_precision", "mean"),
            feature_auc=("feature_auc", "mean"),
            detection_rate=("detected", "mean"),
            n=("sample", "count"),
        )
        .reset_index()
    )
    FA = (
        Fr.groupby(["dataset", "method", "perturbation_mode", "magnitude"])
        .agg(
            p1_mean=("p1", "mean"),
            p1_std=("p1", "std"),
            p3_mean=("p3", "mean"),
            p3_std=("p3", "std"),
            r5_mean=("r5", "mean"),
            mean_rank=("mean_rank", "mean"),
            mrr=("mrr", "mean"),
            ndcg5=("ndcg5", "mean"),
            average_precision=("average_precision", "mean"),
            feature_auc=("feature_auc", "mean"),
            detection_rate=("detection_rate", "mean"),
            n_runs=("fold", "count"),
        )
        .reset_index()
    )
    FA.to_csv(OUT / "tables/table_faithfulness.csv", index=False)
    RT = (
        R.groupby(["dataset", "method"])
        .agg(
            latency_mean=("seconds_per_sample", "mean"),
            latency_median=("seconds_per_sample", "median"),
            latency_std=("seconds_per_sample", "std"),
            n_measurements=("repeat", "count"),
        )
        .reset_index()
    )
    RT.to_csv(OUT / "tables/table_runtime_end_to_end.csv", index=False)
    Cr = (
        C.groupby(["dataset", "seed_idx", "fold", "method"])
        .agg(
            deletion_flip=("deletion_flip", "mean"),
            restricted_k=("restricted_deletion_k", "mean"),
            log_audc=("log_audc", "mean"),
            insertion_flip=("insertion_flip", "mean"),
            restricted_insertion_k=("restricted_insertion_k", "mean"),
            log_auic=("log_auic", "mean"),
            n=("sample", "count"),
        )
        .reset_index()
    )
    CF = (
        Cr.groupby(["dataset", "method"])
        .agg(
            deletion_flip=("deletion_flip", "mean"),
            deletion_flip_std=("deletion_flip", "std"),
            restricted_k=("restricted_k", "mean"),
            restricted_k_std=("restricted_k", "std"),
            log_audc=("log_audc", "mean"),
            log_audc_std=("log_audc", "std"),
            insertion_flip=("insertion_flip", "mean"),
            restricted_insertion_k=("restricted_insertion_k", "mean"),
            log_auic=("log_auic", "mean"),
            n_runs=("fold", "count"),
        )
        .reset_index()
    )
    CF.to_csv(OUT / "tables/table_counterfactual.csv", index=False)
    C.groupby(["dataset", "method", "stratum"]).agg(
        log_audc=("log_audc", "mean"),
        restricted_k=("restricted_deletion_k", "mean"),
        flip_rate=("deletion_flip", "mean"),
        n=("sample", "count"),
    ).reset_index().to_csv(OUT / "tables/table_counterfactual_stratified.csv", index=False)
    C.groupby(["dataset", "method", "subtype"]).agg(
        log_audc=("log_audc", "mean"),
        restricted_k=("restricted_deletion_k", "mean"),
        flip_rate=("deletion_flip", "mean"),
        n=("sample", "count"),
    ).reset_index().to_csv(OUT / "tables/table_counterfactual_per_subtype.csv", index=False)
    V.groupby(["dataset", "subtype"]).agg(
        eligible=("eligible", "sum"),
        detected=("detected", "sum"),
        selected=("selected", "sum"),
        runs=("fold", "count"),
    ).reset_index().to_csv(OUT / "tables/table_subtype_coverage.csv", index=False)
    S.groupby("dataset").agg(
        normal_false_flag=("normal_false_flag", "mean"),
        normal_false_flag_std=("normal_false_flag", "std"),
        anomaly_normal=("anomaly_normal", "mean"),
        anomaly_encoding=("anomaly_encoding", "mean"),
        anomaly_decoding=("anomaly_decoding", "mean"),
        n_runs=("fold", "count"),
    ).reset_index().to_csv(OUT / "tables/table_sared_holdout.csv", index=False)
    for key, out in [
        ("topk", "table_per_attack_topk.csv"),
        ("local", "table_local_explanations.csv"),
        ("sared_distribution", "table_sared_distribution.csv"),
    ]:
        if not Z[key].empty:
            Z[key].to_csv(OUT / "tables" / out, index=False)
    if not Z["feature_behavior"].empty:
        Z["feature_behavior"].groupby(["dataset", "method", "subtype", "feature_idx", "feature_name", "group"]).agg(
            n=("value", "count"),
            mean=("value", "mean"),
            std=("value", "std"),
            median=("value", "median"),
            minimum=("value", "min"),
            maximum=("value", "max"),
        ).reset_index().to_csv(OUT / "tables/table_feature_behavior.csv", index=False)
    if not Z["agreement"].empty:
        Z["agreement"].groupby(["dataset", "pair"]).agg(
            spearman_mean=("spearman_mean", "mean"),
            spearman_std=("spearman_mean", "std"),
            n_runs=("fold", "count"),
        ).reset_index().to_csv(OUT / "tables/table_method_agreement.csv", index=False)
    ST = pd.DataFrame()
    STr = pd.DataFrame()
    if not Z["stability"].empty:
        STr = (
            Z["stability"]
            .groupby(["dataset", "seed_idx", "fold", "method", "regime", "eps"])
            .agg(spearman=("spearman", "mean"), jaccard5=("jaccard5", "mean"), n_samples=("sample", "nunique"))
            .reset_index()
        )
        ST = (
            STr.groupby(["dataset", "method", "regime", "eps"])
            .agg(
                spearman_mean=("spearman", "mean"),
                spearman_std=("spearman", "std"),
                jaccard5_mean=("jaccard5", "mean"),
                jaccard5_std=("jaccard5", "std"),
                n_runs=("fold", "count"),
            )
            .reset_index()
        )
        ST.to_csv(OUT / "tables/table_stability.csv", index=False)
    if not Z["sensitivity"].empty:
        Z["sensitivity"][Z["sensitivity"].kind == "sared_threshold"].groupby(["dataset", "k", "div_pct"]).agg(
            false_flag_rate=("value", "mean"),
            encoding=("encoding", "mean"),
            decoding=("decoding", "mean"),
            n_runs=("fold", "count"),
        ).reset_index().to_csv(OUT / "tables/table_sared_sensitivity.csv", index=False)
        Z["sensitivity"][Z["sensitivity"].kind == "alpha_agreement"].groupby(["dataset", "alpha"]).agg(
            spearman_z_vs_kl=("value", "mean"),
            spearman_std=("value", "std"),
            n_runs=("fold", "count"),
        ).reset_index().to_csv(OUT / "tables/table_alpha_sensitivity.csv", index=False)
    if not Z["suppression"].empty:
        Z["suppression"].groupby(["dataset", "subtype", "metric"]).agg(
            suppression_score=("suppression_score", "mean"),
            suppression_score_std=("suppression_score", "std"),
            excitation_score=("excitation_score", "mean"),
            excitation_score_std=("excitation_score", "std"),
            suppression_ratio=("suppression_ratio", "mean"),
            suppression_ratio_std=("suppression_ratio", "std"),
            n_runs=("fold", "count"),
            n_samples=("n", "sum"),
        ).reset_index().to_csv(OUT / "tables/table_suppression_by_subtype.csv", index=False)
        Z["suppression"].groupby(["dataset", "metric"]).agg(
            suppression_score=("suppression_score", "mean"),
            excitation_score=("excitation_score", "mean"),
            suppression_ratio=("suppression_ratio", "mean"),
            suppression_ratio_std=("suppression_ratio", "std"),
            n_runs_subtypes=("fold", "count"),
            n_samples=("n", "sum"),
        ).reset_index().to_csv(OUT / "tables/table_suppression_by_dataset.csv", index=False)
    if not Z["decoder_transition"].empty:
        DT = Z["decoder_transition"].groupby(["dataset", "from", "to"]).agg(count=("count", "sum")).reset_index()
        DT["pct_of_from"] = DT["count"] / DT.groupby(["dataset", "from"])["count"].transform("sum")
        DT.to_csv(OUT / "tables/table_frozen_decoder_transition.csv", index=False)
        Z["decoder_training"].to_csv(OUT / "tables/table_frozen_decoder_training.csv", index=False)

    # ---- SUPERVISION FIX 9: all-pairs Wilcoxon+Holm over 11 metrics ----
    RUN = []
    Fmax = F[F.magnitude == F.magnitude.max()]  # primary operating point
    fctrl = (
        Fmax[Fmax.perturbation_mode == "scaled_std"]
        .groupby(["dataset", "seed_idx", "fold", "method"])
        .agg(p3=("p3", "mean"), ndcg5=("ndcg5", "mean"), feature_auc=("feature_auc", "mean"))
        .reset_index()
    )
    fdon = (
        F[F.perturbation_mode == "empirical_attack_donor"]
        .groupby(["dataset", "seed_idx", "fold", "method"])
        .agg(p3=("p3", "mean"))
        .reset_index()
    )

    def push(df, col, metric):
        t = df[["dataset", "seed_idx", "fold", "method", col]].rename(columns={col: "value"})
        t["metric"] = metric
        RUN.append(t)

    push(fctrl, "p3", "faith_p3_controlled")
    push(fdon, "p3", "faith_p3_donor")
    push(fctrl, "ndcg5", "faith_ndcg5")
    push(fctrl, "feature_auc", "faith_feature_auc")
    push(Cr, "log_audc", "log_audc")
    push(Cr, "log_auic", "log_auic")
    push(Cr, "restricted_k", "restricted_k")
    push(Cr, "restricted_insertion_k", "restricted_insertion_k")
    if not STr.empty:
        stn = STr[(STr.regime == "input_noise") & (STr.eps == max(CFG["stability_eps"]))]
        push(stn, "spearman", "stability_spearman")
        push(stn, "jaccard5", "stability_jaccard")
    if not R.empty:
        rl = (
            R.groupby(["dataset", "seed_idx", "fold", "method"])
            .agg(seconds_per_sample=("seconds_per_sample", "mean"))
            .reset_index()
        )
        push(rl, "seconds_per_sample", "latency_s")
    stats = []
    for i, t in enumerate(RUN):
        metric = t["metric"].iloc[0]
        stats.extend(all_pairs(t, metric, CFG["seed"] + 100000 * (i + 1)))
    pd.DataFrame(stats).to_csv(OUT / "tables/table_statistics_all_pairs.csv", index=False)

    # suppression x method interaction (SDDA_z vs TSA_adapted)
    P = (
        C[C.method.isin(["SDDA_z", "TSA_adapted"])]
        .pivot_table(index=["dataset", "seed_idx", "fold", "sample", "stratum"], columns="method", values="log_audc")
        .dropna()
    )
    if not P.empty:
        P["difference"] = P["SDDA_z"] - P["TSA_adapted"]
        T = P.reset_index().groupby(["dataset", "seed_idx", "fold", "stratum"]).difference.mean().unstack()
        inter = []
        for ds, g in T.groupby(level=0):
            g = g.droplevel(0)
            if "high" in g and "low" in g:
                d = (g.high - g.low).dropna().values
                lo, hi = ci(d, 999)
                inter.append({"dataset": ds, "high_minus_low_interaction": d.mean(), "ci95_low": lo, "ci95_high": hi, "n_runs": len(d)})
        pd.DataFrame(inter).to_csv(OUT / "tables/table_suppression_interaction.csv", index=False)

    # final benchmark radar source
    FAmax = FA[FA.magnitude == FA.magnitude.max()][["dataset", "method", "p3_mean", "ndcg5", "feature_auc"]]
    FINAL = FAmax.merge(RT[["dataset", "method", "latency_mean"]], on=["dataset", "method"], how="outer")
    if not ST.empty:
        STmax = ST[(ST.regime == "input_noise") & (ST.eps == max(CFG["stability_eps"]))][
            ["dataset", "method", "spearman_mean", "jaccard5_mean"]
        ]
        FINAL = FINAL.merge(STmax, on=["dataset", "method"], how="outer")
    FINAL = FINAL.merge(
        CF[["dataset", "method", "deletion_flip", "restricted_k", "log_audc"]], on=["dataset", "method"], how="outer"
    )
    FINAL["deterministic"] = ~FINAL.method.isin(["KernelSHAP", "Random"])
    FINAL["sared_native"] = FINAL.method.eq("SDDA_z")
    FINAL.to_csv(OUT / "tables/table_final_benchmark.csv", index=False)

    # figure sources
    FA.to_csv(OUT / "figure_sources/fig_faithfulness_source.csv", index=False)
    FINAL.to_csv(OUT / "figure_sources/fig_radar_source.csv", index=False)
    C.groupby(["dataset", "method", "stratum"]).agg(
        log_audc=("log_audc", "mean"), restricted_k=("restricted_deletion_k", "mean"), n=("sample", "count")
    ).reset_index().to_csv(OUT / "figure_sources/fig_counterfactual_stratified_source.csv", index=False)
    if not ST.empty:
        ST.merge(RT, on=["dataset", "method"], how="outer").to_csv(
            OUT / "figure_sources/fig_stability_runtime_source.csv", index=False
        )
    for key, out in [
        ("local", "fig_local_explanation_source.csv"),
        ("feature_behavior", "fig_feature_behavior_source.csv"),
        ("sared_distribution", "fig_sared_source.csv"),
        ("suppression", "fig_suppression_source.csv"),
    ]:
        if not Z[key].empty:
            Z[key].to_csv(OUT / "figure_sources" / out, index=False)

    # feature-schema audit table (supervision fix 5)
    audits = []
    for ds in CFG["datasets"]:
        meta = load_json(OUT / "metadata" / f"{ds}_features.json")
        audits.append(
            {
                "dataset": ds,
                "n_features": meta["n_features"],
                "frame_time_epoch_present": meta["frame_time_epoch_present"],
                "inter_arrival_time_present": meta["inter_arrival_time_present"],
                "feature_hash": meta["hash"],
            }
        )
    pd.DataFrame(audits).to_csv(OUT / "tables/table_artifact_feature_audit.csv", index=False)
    tables = {}
    for path in (OUT / "tables").glob("*.csv"):
        tables[path.stem] = json.loads(pd.read_csv(path).to_json(orient="records"))
    savejson(OUT / "unified_summary_final.json", {"run_hash": RH, "config": CFG, "tables": tables})
    print("Exported:", OUT.resolve())




# Stage execution cells

Jalankan cell di bawah ini satu per satu. Semua stage resume-able karena memakai file output/checkpoint yang sudah disimpan.

In [ ]:
# Stage 0 — preflight + materialize + feature audit
print("=" * 90)
print("STAGE 0 | preflight + materialize |", now())
print("RUN_MODE:", RUN_MODE, "| OUTPUT:", OUT.resolve())
print("=" * 90)

best = preflight()
for ds in CFG["datasets"]:
    print(f"\n##### MATERIALIZE/AUDIT: {ds} #####")
    X, y, ltype, feat_cols = materialize(ds)
    tune_idx, eval_idx, folds = load_splits(ds)
    savejson(
        OUT / "metadata" / f"{ds}_features.json",
        {
            "features": feat_cols,
            "n_features": len(feat_cols),
            "frame_time_epoch_present": "frame.time_epoch" in feat_cols,
            "inter_arrival_time_present": "inter_arrival_time" in feat_cols,
            "hash": hashlib.sha256("|".join(feat_cols).encode()).hexdigest(),
            "n_total_balanced": int(len(y)),
            "n_tune": int(len(tune_idx)),
            "n_eval": int(len(eval_idx)),
        },
    )
    assert len(feat_cols) == CFG["dim"]
    assert "frame.time_epoch" not in feat_cols
    assert "inter_arrival_time" in feat_cols
    del X, y, ltype
    gc.collect()
print("STAGE 0 DONE |", now())


In [ ]:
# Stage 1 — train all configured dataset/seed/fold models
print("=" * 90)
print("STAGE 1 | training |", now())
print("RUN_MODE:", RUN_MODE, "| OUTPUT:", OUT.resolve())
print("=" * 90)

best = preflight()
for ds in CFG["datasets"]:
    print(f"\n##### TRAIN DATASET: {ds} #####")
    X, y, ltype, feat_cols = materialize(ds)
    tune_idx, eval_idx, folds = load_splits(ds)
    X_eval, y_eval = X[eval_idx], y[eval_idx]
    del X, y, ltype
    gc.collect()
    cfg = snn_cfg(ds, best)
    for si in CFG["seeds"]:
        for f in CFG["folds"]:
            train_one(ds, si, f, X_eval, y_eval, folds, cfg)
    del X_eval, y_eval
    gc.collect()
print("STAGE 1 DONE |", now())


In [ ]:
# Stage 2 — build validation-normal baselines for all trained models
print("=" * 90)
print("STAGE 2 | validation-normal baselines |", now())
print("RUN_MODE:", RUN_MODE, "| OUTPUT:", OUT.resolve())
print("=" * 90)

best = preflight()
for ds in CFG["datasets"]:
    print(f"\n##### BASELINE DATASET: {ds} #####")
    X, y, ltype, feat_cols = materialize(ds)
    tune_idx, eval_idx, folds = load_splits(ds)
    X_eval, y_eval = X[eval_idx], y[eval_idx]
    del X, y, ltype
    gc.collect()
    for si in CFG["seeds"]:
        for f in CFG["folds"]:
            build_baseline(ds, si, f, X_eval, y_eval)
    del X_eval, y_eval
    gc.collect()
print("STAGE 2 DONE |", now())


In [ ]:
# Stage 3 — full XAI evaluation for all configured jobs
# This is the heaviest stage because it includes KernelSHAP, stability, counterfactual, and frozen decoder.
print("=" * 90)
print("STAGE 3 | full XAI evaluation |", now())
print("RUN_MODE:", RUN_MODE, "| OUTPUT:", OUT.resolve())
print("=" * 90)

best = preflight()
for ds in CFG["datasets"]:
    print(f"\n##### EVALUATE DATASET: {ds} #####")
    Xp, yp2, lp, cols = pool(ds)
    for si in CFG["seeds"]:
        for f in CFG["folds"]:
            run(ds, si, f, Xp, yp2, lp, cols)
    del Xp, yp2, lp
    gc.collect()
print("STAGE 3 DONE |", now())


In [ ]:
# Stage 4 — aggregate/export final tables and figure sources
print("=" * 90)
print("STAGE 4 | export tables |", now())
print("RUN_MODE:", RUN_MODE, "| OUTPUT:", OUT.resolve())
print("=" * 90)

export()
print("STAGE 4 DONE |", now())


In [ ]:
# Stage 5 — lightweight final audit
print("=" * 90)
print("STAGE 5 | output audit |", now())
print("RUN_MODE:", RUN_MODE, "| OUTPUT:", OUT.resolve())
print("=" * 90)

expected = len(CFG["datasets"]) * len(CFG["seeds"]) * len(CFG["folds"])
model_files = list((OUT / "models").glob("*/*/model.keras"))
baseline_files = list((OUT / "baselines").glob("*/*/base/baseline.npz"))
done_files = list((OUT / "jobs").glob("*/done.json"))
print("Expected jobs:", expected)
print("Models      :", len(model_files))
print("Baselines   :", len(baseline_files))
print("Done evals  :", len(done_files))

required_tables = [
    "table_detection.csv",
    "table_detection_raw.csv",
    "table_split_audit.csv",
    "table_artifact_feature_audit.csv",
    "table_perturbation_effectiveness.csv",
    "table_faithfulness.csv",
    "table_counterfactual.csv",
    "table_runtime_end_to_end.csv",
    "table_sared_holdout.csv",
    "table_sared_distribution.csv",
    "table_per_attack_topk.csv",
    "table_local_explanations.csv",
    "table_feature_behavior.csv",
    "table_stability.csv",
    "table_statistics_all_pairs.csv",
    "table_final_benchmark.csv",
]
missing = []
empty = []
for name in required_tables:
    p = OUT / "tables" / name
    if not p.exists():
        missing.append(name)
    elif p.stat().st_size == 0:
        empty.append(name)
print("Missing tables:", missing)
print("Empty tables  :", empty)

if RUN_MODE == "final":
    assert len(model_files) == expected, "model count mismatch"
    assert len(baseline_files) == expected, "baseline count mismatch"
    assert len(done_files) == expected, "evaluation done count mismatch"
assert not missing, f"missing tables: {missing}"
assert not empty, f"empty tables: {empty}"

feat_audit = pd.read_csv(OUT / "tables/table_artifact_feature_audit.csv")
assert (feat_audit["n_features"] == CFG["dim"]).all()
assert (feat_audit["frame_time_epoch_present"] == False).all()
assert (feat_audit["inter_arrival_time_present"] == True).all()

if (OUT / "tables/table_split_audit.csv").exists():
    sp = pd.read_csv(OUT / "tables/table_split_audit.csv")
    assert (sp["train_attack_count"] == 0).all()
    assert (sp["val_attack_count"] == 0).all()
    assert (sp[["train_val_overlap", "train_test_overlap", "val_test_overlap"]] == 0).all().all()

if (OUT / "tables/table_perturbation_effectiveness.csv").exists():
    pe = pd.read_csv(OUT / "tables/table_perturbation_effectiveness.csv")
    assert (pe["unchanged_slots"] == 0).all()
    assert (pe["zero_std_selected"] == 0).all()

print("AUDIT PASSED |", now())


In [ ]:
# ============================================================
# ZIP IMPORTANT FINAL XAI OUTPUTS ONLY
# Paper 3 — Faithfulness v4, 3 seeds × 10 folds
# ============================================================

from __future__ import annotations

import csv
import hashlib
import json
import shutil
import zipfile
from datetime import datetime
from pathlib import Path

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

ROOT = Path(
    "./"
    "paper3-faithfulness-v4-3x10"
)

NOTEBOOK_CANDIDATES = [
    Path.cwd() / "3-4-paper3_xai_v4_3x10_final_multicell.ipynb",
    Path.cwd() / "3-4-paper3_xai_v4_3x10_final_multicell(1).ipynb",
    ROOT.parent / "3-4-paper3_xai_v4_3x10_final_multicell.ipynb",
]

OUTPUT_DIR = ROOT.parent / "validation-packages"
TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
PACKAGE_NAME = f"paper3_xai_v4_3x10_important_outputs_{TIMESTAMP}"

STAGING = OUTPUT_DIR / PACKAGE_NAME
ZIP_PATH = OUTPUT_DIR / f"{PACKAGE_NAME}.zip"

# File size limit for individual evidence files.
# Important CSV/JSON outputs should normally be far below this.
MAX_FILE_SIZE = 100 * 1024 * 1024  # 100 MB

# ------------------------------------------------------------
# 2. VALIDATION
# ------------------------------------------------------------

if not ROOT.exists():
    raise FileNotFoundError(f"Experiment root not found:\n{ROOT}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if STAGING.exists():
    shutil.rmtree(STAGING)

STAGING.mkdir(parents=True)

# ------------------------------------------------------------
# 3. IMPORTANT DIRECTORY RULES
# ------------------------------------------------------------

# Entire directories normally containing compact final evidence.
FULL_DIRECTORIES = [
    "metadata",
    "tables",
    "figure_sources",
    "figures",
    "audit",
    "audits",
    "summary",
    "summaries",
    "statistics",
    "stats",
    "supplementary",
]

# Per-run files needed to verify values without copying model weights.
PER_RUN_ALLOWED_NAMES = {
    "done.json",
    "config.json",
    "run_config.json",
    "run_metadata.json",
    "metrics.json",
    "summary.json",
    "threshold.json",
    "detection_sanity.json",
    "baseline_summary.json",
    "split_audit.csv",
    "history.csv",
    "detection.csv",
    "metrics.csv",
    "runtime.csv",
    "stability.csv",
    "faithfulness.csv",
    "deletion.csv",
    "insertion.csv",
    "sared.csv",
    "suppression.csv",
    "local_explanation.csv",
}

# Extensions allowed from compact evidence directories and jobs.
ALLOWED_EXTENSIONS = {
    ".csv",
    ".tsv",
    ".json",
    ".txt",
    ".md",
    ".yaml",
    ".yml",
    ".npz",
    ".npy",
    ".png",
    ".pdf",
    ".svg",
}

# Large/nonessential artifacts excluded.
EXCLUDED_EXTENSIONS = {
    ".keras",
    ".h5",
    ".hdf5",
    ".pkl",
    ".pickle",
    ".joblib",
    ".parquet",
    ".gz",
    ".zip",
    ".tar",
    ".7z",
}

EXCLUDED_PARTS = {
    "__pycache__",
    ".ipynb_checkpoints",
    "cache",
    "caches",
    "dataset",
    "datasets",
    "raw_dataset",
    "raw-data",
    "checkpoints",
}

# ------------------------------------------------------------
# 4. HELPERS
# ------------------------------------------------------------

copied_records: list[dict] = []
skipped_records: list[dict] = []


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()


def is_excluded(path: Path) -> bool:
    lower_parts = {part.lower() for part in path.parts}

    if lower_parts.intersection(EXCLUDED_PARTS):
        return True

    if path.suffix.lower() in EXCLUDED_EXTENSIONS:
        return True

    return False


def copy_evidence(source: Path, relative_target: Path, category: str) -> None:
    if not source.is_file():
        return

    if is_excluded(source):
        skipped_records.append(
            {
                "source": str(source),
                "reason": "excluded path or extension",
            }
        )
        return

    size = source.stat().st_size

    if size > MAX_FILE_SIZE:
        skipped_records.append(
            {
                "source": str(source),
                "reason": f"file larger than {MAX_FILE_SIZE:,} bytes",
            }
        )
        return

    destination = STAGING / relative_target
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)

    copied_records.append(
        {
            "relative_path": relative_target.as_posix(),
            "category": category,
            "size_bytes": size,
            "sha256": sha256_file(destination),
        }
    )


def copy_directory(directory_name: str) -> None:
    source_dir = ROOT / directory_name

    if not source_dir.exists():
        return

    for source in sorted(source_dir.rglob("*")):
        if not source.is_file():
            continue

        if source.suffix.lower() not in ALLOWED_EXTENSIONS:
            continue

        relative = source.relative_to(ROOT)
        copy_evidence(source, relative, f"directory:{directory_name}")


# ------------------------------------------------------------
# 5. COPY FINAL TABLES, STATISTICS, FIGURE SOURCES, METADATA
# ------------------------------------------------------------

for directory_name in FULL_DIRECTORIES:
    copy_directory(directory_name)

# ------------------------------------------------------------
# 6. COPY IMPORTANT ROOT-LEVEL FILES
# ------------------------------------------------------------

ROOT_LEVEL_KEYWORDS = (
    "audit",
    "manifest",
    "config",
    "metadata",
    "summary",
    "signature",
    "checksum",
    "readme",
    "environment",
    "requirements",
    "protocol",
)

for source in sorted(ROOT.iterdir()):
    if not source.is_file():
        continue

    name_lower = source.name.lower()

    if source.suffix.lower() not in ALLOWED_EXTENSIONS:
        continue

    if any(keyword in name_lower for keyword in ROOT_LEVEL_KEYWORDS):
        copy_evidence(
            source,
            Path("root_files") / source.name,
            "root-level evidence",
        )

# ------------------------------------------------------------
# 7. COPY COMPACT PER-JOB OUTPUTS
# ------------------------------------------------------------

jobs_dir = ROOT / "jobs"

if jobs_dir.exists():
    for source in sorted(jobs_dir.rglob("*")):
        if not source.is_file():
            continue

        name_lower = source.name.lower()
        suffix = source.suffix.lower()

        include = (
            source.name in PER_RUN_ALLOWED_NAMES
            or suffix in {".csv", ".json", ".txt", ".tsv"}
            and any(
                keyword in name_lower
                for keyword in (
                    "metric",
                    "faith",
                    "donor",
                    "deletion",
                    "insertion",
                    "stability",
                    "runtime",
                    "sared",
                    "suppression",
                    "local",
                    "pairwise",
                    "wilcoxon",
                    "holm",
                    "bootstrap",
                    "summary",
                    "audit",
                )
            )
        )

        if include:
            relative = source.relative_to(ROOT)
            copy_evidence(source, relative, "per-job evidence")

# ------------------------------------------------------------
# 8. COPY MODEL SUMMARIES ONLY — NO MODEL WEIGHTS
# ------------------------------------------------------------

models_dir = ROOT / "models"

if models_dir.exists():
    for source in sorted(models_dir.rglob("*")):
        if not source.is_file():
            continue

        if source.name in {
            "detection.csv",
            "detection_sanity.json",
            "split_audit.csv",
            "threshold.json",
            "history.csv",
            "config.json",
            "run_metadata.json",
        }:
            relative = source.relative_to(ROOT)
            copy_evidence(source, relative, "model summary")

# ------------------------------------------------------------
# 9. COPY BASELINE SUMMARIES
# ------------------------------------------------------------

baselines_dir = ROOT / "baselines"

if baselines_dir.exists():
    for source in sorted(baselines_dir.rglob("*")):
        if not source.is_file():
            continue

        if source.suffix.lower() in {".csv", ".json", ".txt"}:
            relative = source.relative_to(ROOT)
            copy_evidence(source, relative, "baseline summary")

        # Include baseline.npz only when compact enough.
        elif source.name == "baseline.npz":
            relative = source.relative_to(ROOT)
            copy_evidence(source, relative, "validation-normal baseline")

# ------------------------------------------------------------
# 10. COPY NOTEBOOK SOURCE WHEN FOUND
# ------------------------------------------------------------

notebook_found = None

for candidate in NOTEBOOK_CANDIDATES:
    if candidate.exists():
        notebook_found = candidate
        copy_evidence(
            candidate,
            Path("source") / candidate.name,
            "experiment notebook",
        )
        break

# ------------------------------------------------------------
# 11. PACKAGE METADATA
# ------------------------------------------------------------

package_info = {
    "package_name": PACKAGE_NAME,
    "created_at": datetime.now().isoformat(),
    "experiment_root": str(ROOT),
    "experiment_identity": {
        "run_mode": "final",
        "expected_jobs": 60,
        "expected_models": 60,
        "expected_baselines": 60,
        "expected_completed_evaluations": 60,
        "reported_audit_status": "AUDIT PASSED",
        "audit_timestamp": "2026-08-01 13:03:33",
    },
    "scope": (
        "Compact manuscript-validation package containing final tables, "
        "statistics, metadata, figure sources, compact per-run outputs, "
        "model summaries, validation-normal baselines, and source notebook."
    ),
    "excluded_by_design": [
        "trained .keras/.h5 model weights",
        "scaler .pkl/.pickle files",
        "raw datasets",
        "large checkpoints",
        "cache directories",
        "existing archive files",
    ],
    "notebook_included": str(notebook_found) if notebook_found else None,
    "file_count": len(copied_records),
    "total_uncompressed_bytes": sum(
        record["size_bytes"] for record in copied_records
    ),
}

with (STAGING / "PACKAGE_INFO.json").open("w", encoding="utf-8") as handle:
    json.dump(package_info, handle, indent=2, ensure_ascii=False)

# ------------------------------------------------------------
# 12. WRITE FILE MANIFEST
# ------------------------------------------------------------

manifest_path = STAGING / "FILE_MANIFEST.csv"

with manifest_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "relative_path",
            "category",
            "size_bytes",
            "sha256",
        ],
    )
    writer.writeheader()
    writer.writerows(
        sorted(copied_records, key=lambda item: item["relative_path"])
    )

# ------------------------------------------------------------
# 13. WRITE SHA256SUMS
# ------------------------------------------------------------

checksum_path = STAGING / "SHA256SUMS.txt"

with checksum_path.open("w", encoding="utf-8") as handle:
    for record in sorted(
        copied_records,
        key=lambda item: item["relative_path"],
    ):
        handle.write(
            f'{record["sha256"]}  {record["relative_path"]}\n'
        )

# ------------------------------------------------------------
# 14. WRITE SKIPPED FILE REPORT
# ------------------------------------------------------------

skipped_path = STAGING / "SKIPPED_FILES.csv"

with skipped_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["source", "reason"],
    )
    writer.writeheader()
    writer.writerows(skipped_records)

# ------------------------------------------------------------
# 15. VALIDATION SUMMARY
# ------------------------------------------------------------

required_candidates = {
    "tables": ROOT / "tables",
    "metadata": ROOT / "metadata",
    "jobs": ROOT / "jobs",
}

validation_summary = {
    "required_locations": {
        name: {
            "path": str(path),
            "exists": path.exists(),
        }
        for name, path in required_candidates.items()
    },
    "copied_files": len(copied_records),
    "skipped_files": len(skipped_records),
    "notebook_found": notebook_found is not None,
}

with (STAGING / "VALIDATION_SUMMARY.json").open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(validation_summary, handle, indent=2)

# ------------------------------------------------------------
# 16. CREATE ZIP
# ------------------------------------------------------------

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True,
) as archive:
    for source in sorted(STAGING.rglob("*")):
        if source.is_file():
            archive_name = (
                Path(PACKAGE_NAME) / source.relative_to(STAGING)
            )
            archive.write(source, archive_name.as_posix())

# ------------------------------------------------------------
# 17. FINAL ZIP CHECKSUM
# ------------------------------------------------------------

zip_sha256 = sha256_file(ZIP_PATH)
zip_size_mb = ZIP_PATH.stat().st_size / (1024 * 1024)

zip_checksum_file = ZIP_PATH.with_suffix(".zip.sha256")
zip_checksum_file.write_text(
    f"{zip_sha256}  {ZIP_PATH.name}\n",
    encoding="utf-8",
)

# Remove staging after successful archive creation.
shutil.rmtree(STAGING)

print("=" * 90)
print("IMPORTANT XAI VALIDATION PACKAGE CREATED")
print("=" * 90)
print(f"Experiment root : {ROOT}")
print(f"Files included  : {len(copied_records):,}")
print(f"Files skipped   : {len(skipped_records):,}")
print(f"Notebook        : {notebook_found or 'NOT FOUND'}")
print(f"ZIP             : {ZIP_PATH}")
print(f"ZIP size        : {zip_size_mb:,.2f} MB")
print(f"ZIP SHA-256     : {zip_sha256}")
print(f"Checksum file   : {zip_checksum_file}")
print("=" * 90)